# 02 · Step 3 runs: calibrated confidence (Track A)
**Advanced GenAI HS26 · Trustworthy Agentic RAG · Group 2: Divya Shori, Faranak Rahimi, Hanieh Jebeli**

What this notebook does:
1. runs the **unchanged reference system** on the instructor's **development benchmark (80 questions)** and the **original 25 questions**, `N_RUNS` times;
2. grades every answer with the **reference team's own judge prompt** (Haiku);
3. trains and evaluates our **calibrator + abstention gate** leave-one-question-out, with the ablation ladder, and saves tables and plots to Drive.

Everything is saved to `My Drive/AdvGenAI/step3/` as it goes. If Colab disconnects, run all again: finished questions are skipped.
Cost estimate (Opus generation, as in Step 1): about **$3–4 per run of 105 questions**, about 20–25 min per run.

**The held-out test set is not used anywhere.** The system only receives the question text (no labels or notes).

## 1 · Setup (same as notebooks 00 and 01)

In [ ]:
# ---- edit if your Drive layout differs --------------------------------------
WORK_DIR_NAME   = "AdvGenAI"                 # our group folder in My Drive (outputs go here)
ARTIFACT_FOLDER = "RAG_Project_Artifact"     # name of the shortcut to the shared folder
COPY_TO_LOCAL   = True    # copy indexes to Colab disk: faster, and avoids Chroma writing to a read-only folder

# Models: the reference run used these. Keep them for reproduction; a cheaper
# generator model is fine for exploratory runs, but note it in the log.
AGENT_MODEL_GENERATION = "claude-opus-4-7"
AGENT_MODEL_VERIFIER   = "claude-opus-4-7"
AGENT_MODEL_UTILITY    = "claude-haiku-4-5"
LIVE = True    # False = replay + index checks only (no model download, no API cost)


In [ ]:
import os, sys, json, time, shutil, subprocess, traceback, pathlib, platform, datetime
ISSUES = []
def log_issue(step, err=None, note=""):
    tb = traceback.format_exc() if err is not None else ""
    ISSUES.append({"time": datetime.datetime.now().isoformat(timespec="seconds"),
                   "step": step, "error": repr(err) if err else "", "note": note, "traceback": tb[-3000:]})
    print(f"⚠️  logged issue at [{step}]: {err or note}")

def step(name):
    # Use as: with step("x"): ...  → exceptions are logged, not raised
    class _S:
        def __enter__(self): print(f"\n▶ {name}"); self.t = time.time()
        def __exit__(self, et, ev, tb):
            if ev is not None:
                log_issue(name, ev); return True
            print(f"  ✓ {name} ({time.time()-self.t:.0f}s)")
    return _S()

print("Python", platform.python_version())
try:
    import torch
    print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
    if not torch.cuda.is_available():
        log_issue("gpu", note="No GPU runtime: model loading/retrieval will be very slow on CPU.")
except Exception as e:
    log_issue("torch import", e)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
MYDRIVE = pathlib.Path("/content/drive/MyDrive")
WORK = MYDRIVE / WORK_DIR_NAME
WORK.mkdir(parents=True, exist_ok=True)

def find_one(pattern, roots, maxdepth=4):
    for root in roots:
        for depth in range(maxdepth + 1):
            hits = list(root.glob("/".join(["*"] * depth + [pattern])) if depth else root.glob(pattern))
            if hits: return hits[0]
    return None

with step("locate shared full_corpus"):
    FULL_CORPUS_SRC = None
    cand = MYDRIVE / ARTIFACT_FOLDER / "storage" / "full_corpus"
    FULL_CORPUS_SRC = cand if cand.is_dir() else find_one("storage/full_corpus", [MYDRIVE], 3)
    assert FULL_CORPUS_SRC, ("full_corpus not found. Did you add the RAG_Project_Artifact shortcut to My Drive?")
    print("  full_corpus:", FULL_CORPUS_SRC)
    for sub in ["retrieval/fixed_size_chunk/bm25_retriever_full.pkl", "vectordb_dense/fixed_e5/chroma.sqlite3",
                "retrieval_graph/comm2chunk_fixed.json", "retrieval_graph/embeddings", "Lang_norm/fixed_size_chunk"]:
        p = FULL_CORPUS_SRC / sub
        print(f"   {'OK ' if p.exists() else '!! '} {sub}")
        if not p.exists(): log_issue("artifact check", note=f"missing in shared folder: {sub}")

with step("locate Baseline.zip"):
    ZIP = WORK / "Baseline.zip"
    ZIP = ZIP if ZIP.exists() else find_one("Baseline.zip", [MYDRIVE], 3)
    print("  Baseline.zip:", ZIP)

In [ ]:
REPO = pathlib.Path("/content/Baseline")          # repo root (contains src/, results/, Baseline/benchmark)
with step("unpack reference code"):
    if not (REPO / "src").exists():
        if ZIP:
            subprocess.run(["unzip", "-q", "-o", str(ZIP), "-d", "/content"], check=True)
        else:
            log_issue("code source", note="Baseline.zip not found, so cloning GitHub (no qrels: retrieval metrics unavailable)")
            subprocess.run(["git", "clone", "-q", "https://github.com/CyrielVanHelleputte/Adv.-Gen-AI", str(REPO)], check=True)
    assert (REPO / "src" / "agentic" / "pipeline.py").exists(), "unexpected repo layout"
    print("  repo:", REPO)

BASELINE_ROOT = REPO / "Baseline"                 # what config.py expects
STORAGE_LINK  = BASELINE_ROOT / "storage" / "full_corpus"
with step("stage full_corpus"):
    STORAGE_LINK.parent.mkdir(parents=True, exist_ok=True)
    if STORAGE_LINK.is_symlink() or STORAGE_LINK.exists():
        print("  already staged:", STORAGE_LINK.resolve())
    elif COPY_TO_LOCAL:
        local = pathlib.Path("/content/full_corpus")
        if not local.exists():
            print("  copying shared indexes to local disk (a few minutes)...")
            subprocess.run(["rsync", "-a", "--exclude", "__pycache__", f"{FULL_CORPUS_SRC}/", f"{local}/"], check=True)
        STORAGE_LINK.symlink_to(local)
    else:
        STORAGE_LINK.symlink_to(FULL_CORPUS_SRC)
    print("  du:", subprocess.run(["du", "-sh", str(STORAGE_LINK.resolve())], capture_output=True, text=True).stdout.strip())

In [ ]:
%%capture pip_out
!pip install -q "langchain>=0.3,<0.4" "langchain-community>=0.3,<0.4" "langchain-core>=0.3,<0.4" \
    "langchain-huggingface>=0.1,<0.4" chromadb "rank_bm25>=0.2.2" "sentence-transformers>=3.0" \
    "pytrec_eval-terrier>=0.5.7" langdetect sentencepiece "anthropic>=0.69" python-dotenv

In [ ]:
tail = pip_out.stdout[-1500:] + pip_out.stderr[-1500:]
if "ERROR" in tail: log_issue("pip install", note=tail)
from importlib.metadata import version
VERSIONS = {p: version(p) for p in ["langchain","langchain-community","langchain-core","langchain-huggingface",
            "chromadb","sentence-transformers","transformers","torch","anthropic","numpy"]}
print(json.dumps(VERSIONS, indent=1))

In [ ]:
from google.colab import userdata
with step("read ANTHROPIC_API_KEY secret"):
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["BASELINE_ROOT"] = str(BASELINE_ROOT)
os.environ["AGENT_MODEL_GENERATION"] = AGENT_MODEL_GENERATION
os.environ["AGENT_MODEL_VERIFIER"]   = AGENT_MODEL_VERIFIER
os.environ["AGENT_MODEL_UTILITY"]    = AGENT_MODEL_UTILITY

os.chdir(REPO)
SRC = str(REPO / "src")
if SRC not in sys.path: sys.path.insert(0, SRC)


In [ ]:
from baseline import config
from agentic import AgenticRAG
with step("AgenticRAG.from_pretrained()"):
    print("  models:", config.AGENT_MODEL_GENERATION, config.AGENT_MODEL_VERIFIER, config.AGENT_MODEL_UTILITY)
    rag = AgenticRAG.from_pretrained()


## 2 · Our Step 3 code (embedded from the repo, `src/ours/improvement`)

In [ ]:
FILES = {"g02step3/__init__.py": "", "g02step3/benchmarks.py": "\"\"\"Load the benchmarks used in Step 3 into one common format.\n\nEach item is a dict with:\n    id, question, category, expected (primary final outcome),\n    acceptable (set of acceptable final outcomes, lower case),\n    reference (gold / reference answer or None)\n\nTwo sources:\n* the instructor's development benchmark (80 questions, JSONL), and\n* the original 25 questions with their ORIGINAL labels (reliability_labels.json),\n  which must be used unchanged (instructor decision, 19 Nov).\n\"\"\"\nfrom __future__ import annotations\n\nimport json\nimport os\nimport pathlib\n\n# Repo root; in Colab the notebook sets G02_ROOT to where it wrote the data files.\nROOT = pathlib.Path(os.environ.get(\"G02_ROOT\", pathlib.Path(__file__).resolve().parents[3]))\nDEV_JSONL = ROOT / \"benchmark\" / \"dev\" / \"Development_Benchmark_v1.0.jsonl\"\nORIGINAL_LABELS = ROOT / \"reference\" / \"Baseline\" / \"benchmark\" / \"reliability_labels.json\"\n\n\ndef _clean_ref(text):\n    if not text:\n        return None\n    t = str(text).strip()\n    # \"(No answer: ...)\" marks abstention cases: there is no gold answer to grade against.\n    return None if t.startswith(\"(No answer\") else t\n\n\ndef load_dev(path=None) -> list[dict]:\n    path = path or DEV_JSONL\n    items = []\n    for line in pathlib.Path(path).read_text(encoding=\"utf-8\").splitlines():\n        if not line.strip():\n            continue\n        r = json.loads(line)\n        acc = {a.strip().lower() for a in str(r[\"acceptable_final_outcomes\"]).split(\"|\") if a.strip()}\n        items.append({\n            \"id\": r[\"case_id\"],\n            \"question\": r[\"question\"],\n            \"category\": r[\"category\"],\n            \"group\": r[\"group_id\"],\n            \"expected\": r[\"expected_final_outcome\"].lower(),\n            \"acceptable\": acc,\n            \"reference\": _clean_ref(r.get(\"reference_answer\")),\n        })\n    return items\n\n\ndef load_original(path=None) -> list[dict]:\n    path = path or ORIGINAL_LABELS\n    labels = json.loads(pathlib.Path(path).read_text(encoding=\"utf-8\"))\n    items = []\n    for qid in sorted(labels, key=int):\n        L = labels[qid]\n        exp = L[\"expected_behavior\"].lower()\n        items.append({\n            \"id\": str(qid),\n            \"question\": L[\"question\"],\n            \"category\": \"original\",\n            \"group\": \"original\",\n            \"expected\": exp,\n            \"acceptable\": {exp},\n            \"reference\": L.get(\"gold_answer\"),\n        })\n    return items\n\n\ndef to_qa(items: list[dict]) -> list[dict]:\n    \"\"\"Items for the reference system's run_benchmark(): id + question ONLY.\n\n    No category, notes or labels are passed, so the system cannot see the answer key\n    (the reference profiler reads ambiguity hints from qa_item['notes'] otherwise).\n    \"\"\"\n    return [{\"id\": it[\"id\"], \"question\": it[\"question\"]} for it in items]\n", "g02step3/features.py": "\"\"\"Turn one decision trace into calibration features.\n\nFeature groups (used for the ablation; each group adds to the previous ones):\n\n  G0  baseline      the reference system's own confidence score\n  G1  signals       final sufficiency, chunk agreement, grounding, contradiction,\n                    number of unsupported claims\n  G2  process       how the system got there: retrieval rounds, recoveries, mode changes,\n                    generator 'no evidence' rounds, sufficiency trajectory, ambiguity flag,\n                    question language, latency\n  G3  answer text   does the answer actually answer? hand-written hedge / non-answer\n                    phrases, answer length, number of citations\n  G4  self-check    one cheap LLM call per answer (no gold answer): \"does this answer\n                    actually answer the question?\" yes / partly / no\n\nG0-G3 are read from the saved trace (no extra calls). G4 needs selfcheck.json from the\nrun notebook; without it the feature is neutral (0.5).\n\nThe hedge list in G3 was written after looking at the Step 1 failures, so the ablation\nalso reports a ladder WITHOUT it (G0+G1+G2+G4) to show the result does not depend on it.\n\"\"\"\nfrom __future__ import annotations\n\nimport re\n\n# Phrases that signal the 'answer' is really a non-answer (Step 1, Q10/Q12/Q20).\n_HEDGES = [\n    r\"do(?:es)? not (?:contain|mention|provide|give|specify|state|include|say|identify|describe|cover)\",\n    r\"no (?:specific |explicit |direct )?(?:information|mention|data|details|evidence)\",\n    r\"not (?:clearly |explicitly )?(?:stated|mentioned|specified|provided|available|covered)\",\n    r\"(?:cannot|can't|unable to) (?:determine|confirm|find|identify|answer)\",\n    r\"(?:only )?partially (?:covers?|answer)\",\n    r\"the (?:provided |available )?(?:chunks|evidence|context|sources) (?:do not|don't|does not)\",\n    r\"unclear\",\n    r\"insufficient\",\n]\n_HEDGE_RE = [re.compile(p, re.I) for p in _HEDGES]\n\nGROUPS = {\n    \"G0\": [\"base_conf\"],\n    \"G1\": [\"suff\", \"agree\", \"ground\", \"contra\", \"n_unsupported\"],\n    \"G2\": [\"n_retrieve\", \"n_recover\", \"mode_changed\", \"n_no_evidence\", \"suff_first\",\n           \"suff_gain\", \"ambiguous\", \"lang_de\", \"latency_s\"],\n    \"G3\": [\"n_hedges\", \"has_hedge\", \"answer_len_log\", \"n_citations\"],\n    \"G4\": [\"selfcheck\"],\n}\nG2 = GROUPS[\"G2\"] + [\"ret_jaccard\", \"cite_share\"]\nGROUPS[\"G2\"] = G2\n\nSELFCHECK_SCORE = {\"yes\": 1.0, \"partly\": 0.5, \"no\": 0.0}\n\n\ndef _steps(trace, name):\n    return [s for s in trace.get(\"timeline\", []) if s.get(\"step\") == name]\n\n\ndef _jaccard(a, b):\n    a, b = set(a or []), set(b or [])\n    return len(a & b) / len(a | b) if (a or b) else 1.0\n\n\ndef features(trace: dict, selfcheck: dict | None = None) -> dict:\n    fin = trace.get(\"final\", {}) or {}\n    sig = fin.get(\"signals\") or {}\n    analyses = [s.get(\"signals\", {}) for s in _steps(trace, \"analyze\")]\n    verifies = _steps(trace, \"verify\")\n    gens = _steps(trace, \"generate\")\n    rets = _steps(trace, \"retrieve\")\n    prof = trace.get(\"profile\", {}) or {}\n    ans = fin.get(\"answer\") or \"\"\n    suff_series = [a.get(\"sufficiency\", 0.0) or 0.0 for a in analyses] or [0.0]\n    last_v = verifies[-1] if verifies else {}\n    hedges = sum(len(r.findall(ans)) for r in _HEDGE_RE)\n    t_ms = max([s.get(\"t_ms\", 0) or 0 for s in trace.get(\"timeline\", [])] + [0])\n    import math\n\n    return {\n        \"base_conf\": float(fin.get(\"confidence\") or 0.0),\n        \"suff\": float(sig.get(\"sufficiency\", suff_series[-1]) or 0.0),\n        \"agree\": float(sig.get(\"chunk_agreement\", 0.0) or 0.0),\n        \"ground\": float(sig.get(\"grounding\", last_v.get(\"grounding_score\") or 0.0) or 0.0),\n        \"contra\": float(sig.get(\"contradiction\", last_v.get(\"contradiction_score\") or 0.0) or 0.0),\n        \"n_unsupported\": float(len(last_v.get(\"unsupported_claims\") or [])),\n        \"n_retrieve\": float(len(rets)),\n        \"n_recover\": float(len(_steps(trace, \"recover\"))),\n        \"mode_changed\": float(len({r.get(\"mode\") for r in rets}) > 1),\n        \"n_no_evidence\": float(sum(1 for g in gens if g.get(\"evidence_found\") is False)),\n        \"suff_first\": float(suff_series[0]),\n        \"suff_gain\": float(suff_series[-1] - suff_series[0]),\n        \"ambiguous\": float(bool(prof.get(\"ambiguity_flag\"))),\n        \"lang_de\": float(prof.get(\"lang\") == \"de\"),\n        \"latency_s\": float(t_ms) / 1000.0,\n        \"n_hedges\": float(hedges),\n        \"has_hedge\": float(hedges > 0),\n        \"answer_len_log\": math.log1p(len(ans.split())),\n        \"n_citations\": float(len(fin.get(\"citations\") or [])),\n        \"ret_jaccard\": _jaccard(rets[0].get(\"top_ids\"), rets[-1].get(\"top_ids\")) if rets else 1.0,\n        \"cite_share\": (len(set(fin.get(\"citations\") or []) & set(rets[-1].get(\"top_ids\") or []))\n                       / max(1, len(rets[-1].get(\"top_ids\") or []))) if rets else 0.0,\n        \"selfcheck\": SELFCHECK_SCORE.get((selfcheck or {}).get(\"answers_question\"), 0.5),\n    }\n\n\nLADDERS = {\n    \"G1\": [\"G0\", \"G1\"],\n    \"G2\": [\"G0\", \"G1\", \"G2\"],\n    \"G3\": [\"G0\", \"G1\", \"G2\", \"G3\"],\n    \"G4\": [\"G0\", \"G1\", \"G2\", \"G3\", \"G4\"],\n    \"noHedge\": [\"G0\", \"G1\", \"G2\", \"G4\"],      # without the hand-written hedge list\n}\n\n\ndef columns(ladder: str) -> list[str]:\n    \"\"\"Feature names for one rung of the ablation ladder (see LADDERS).\"\"\"\n    out = []\n    for g in LADDERS[ladder]:\n        out += [c for c in GROUPS[g] if not (ladder == \"noHedge\" and c in (\"n_hedges\", \"has_hedge\"))]\n    return out\n", "g02step3/outcomes.py": "\"\"\"Ground truth for calibration: was a given output acceptable, and how good was it?\n\nFor every (run, question) row:\n\n* y      graded correctness of an ANSWER (1 correct, 0.5 partially correct, 0 incorrect),\n         or 0 when the benchmark does not accept ANSWER for this question\n         (e.g. an abstain or clarify case that the system answered anyway).\n         None when ANSWER is acceptable but there is no reference answer to grade against.\n* score_answer / score_abstain   the row's value if the system answers / abstains.\n         These drive the abstention gate and the end-to-end reliability score.\n\nCorrectness comes from the reference team's LLM judge prompt (same wording, same\nutility model), so our numbers stay comparable with theirs.\n\"\"\"\nfrom __future__ import annotations\n\nfrom typing import Callable, Optional\n\nVERDICT_SCORE = {\"correct\": 1.0, \"partially_correct\": 0.5, \"incorrect\": 0.0}\n\n# Identical to reference/src/scripts/evaluate_agentic.py (_JUDGE_SYSTEM / _JUDGE_SCHEMA).\nJUDGE_SYSTEM = (\n    \"You grade a candidate answer against a gold answer for an ETH Zurich QA \"\n    \"benchmark. Return 'correct' if the candidate contains the gold answer, \"\n    \"'partially_correct' if it is on the right track but incomplete or hedged, \"\n    \"and 'incorrect' otherwise. Judge meaning, not wording.\"\n)\nJUDGE_SCHEMA = {\n    \"type\": \"object\",\n    \"properties\": {\n        \"verdict\": {\"type\": \"string\", \"enum\": [\"correct\", \"partially_correct\", \"incorrect\"]},\n        \"reason\": {\"type\": \"string\"},\n    },\n    \"required\": [\"verdict\", \"reason\"],\n    \"additionalProperties\": False,\n}\n\n\ndef needs_judging(item: dict, trace: dict) -> bool:\n    return (trace[\"final\"].get(\"action\") == \"answer\"\n            and \"answer\" in item[\"acceptable\"] and bool(item.get(\"reference\")))\n\n\ndef judge_run(items: list[dict], traces: dict, grade: Callable[[str, str, str], dict],\n              cache: Optional[dict] = None) -> dict:\n    \"\"\"grade(question, gold, candidate) -> {'verdict', 'reason'}. Returns {qid: {...}}.\"\"\"\n    out = dict(cache or {})\n    for it in items:\n        t = traces.get(it[\"id\"])\n        if t is None or it[\"id\"] in out or not needs_judging(it, t):\n            continue\n        out[it[\"id\"]] = grade(it[\"question\"], it[\"reference\"], t[\"final\"].get(\"answer\") or \"\")\n    return out\n\n\ndef outcome(item: dict, trace: dict, verdict: Optional[dict]) -> dict:\n    action = trace[\"final\"].get(\"action\")\n    acc = item[\"acceptable\"]\n    if \"answer\" not in acc:\n        y = 0.0\n    elif not item.get(\"reference\"):\n        y = None                        # acceptable to answer, nothing to grade against\n    elif verdict is not None:\n        y = VERDICT_SCORE[verdict[\"verdict\"]]\n    else:\n        y = None\n    score_answer = (1.0 if y is None else y) if \"answer\" in acc else 0.0\n    score_abstain = 1.0 if \"abstain\" in acc else 0.0\n    return {\n        \"action\": action,\n        \"action_ok\": action in acc,\n        \"y\": y if action == \"answer\" else None,\n        \"score_answer\": score_answer,\n        \"score_abstain\": score_abstain,\n        \"score_as_is\": score_answer if action == \"answer\" else (1.0 if action in acc else 0.0),\n    }\n\n\n# ---------------------------------------------------------------- G4 self-check\n# Part of the MECHANISM (runs at answer time): it sees only the question and the\n# system's own answer, never a gold answer.\nSELFCHECK_SYSTEM = (\n    \"You check a question-answering system's output. Given a question and the system's \"\n    \"answer, decide whether the answer actually answers the question. Return 'yes' if it \"\n    \"gives a direct, specific answer; 'partly' if it answers only part of the question or \"\n    \"answers with heavy hedging; 'no' if it mainly says the information is missing, \"\n    \"unclear or not found, or answers a different question. Do not judge whether the \"\n    \"answer is factually true.\"\n)\nSELFCHECK_SCHEMA = {\n    \"type\": \"object\",\n    \"properties\": {\n        \"answers_question\": {\"type\": \"string\", \"enum\": [\"yes\", \"partly\", \"no\"]},\n        \"reason\": {\"type\": \"string\"},\n    },\n    \"required\": [\"answers_question\", \"reason\"],\n    \"additionalProperties\": False,\n}\n\n\ndef selfcheck_run(items: list[dict], traces: dict, check: Callable[[str, str], dict],\n                  cache: Optional[dict] = None) -> dict:\n    \"\"\"check(question, answer) -> {'answers_question', 'reason'}, for every ANSWER.\"\"\"\n    out = dict(cache or {})\n    for it in items:\n        t = traces.get(it[\"id\"])\n        if t is None or it[\"id\"] in out or t[\"final\"].get(\"action\") != \"answer\":\n            continue\n        out[it[\"id\"]] = check(it[\"question\"], t[\"final\"].get(\"answer\") or \"\")\n    return out\n", "g02step3/metrics.py": "\"\"\"Calibration and selective-answering metrics.\n\nAll functions take arrays over ANSWERED rows: p = predicted confidence, y = graded\ncorrectness (1 / 0.5 / 0).\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef ece(p, y, n_bins: int = 10) -> float:\n    \"\"\"Expected calibration error, equal-width bins, graded correctness.\"\"\"\n    p, y = np.asarray(p, float), np.asarray(y, float)\n    if len(p) == 0:\n        return float(\"nan\")\n    bins = np.minimum((p * n_bins).astype(int), n_bins - 1)\n    err = 0.0\n    for b in range(n_bins):\n        m = bins == b\n        if m.any():\n            err += m.mean() * abs(p[m].mean() - y[m].mean())\n    return float(err)\n\n\ndef brier(p, y) -> float:\n    p, y = np.asarray(p, float), np.asarray(y, float)\n    return float(np.mean((p - y) ** 2)) if len(p) else float(\"nan\")\n\n\ndef auroc(p, y) -> float:\n    \"\"\"Can confidence separate fully correct answers (y == 1) from the rest?\"\"\"\n    p, y = np.asarray(p, float), np.asarray(y, float)\n    pos, neg = p[y == 1.0], p[y < 1.0]\n    if len(pos) == 0 or len(neg) == 0:\n        return float(\"nan\")\n    wins = (pos[:, None] > neg[None, :]).sum() + 0.5 * (pos[:, None] == neg[None, :]).sum()\n    return float(wins / (len(pos) * len(neg)))\n\n\ndef risk_coverage(p, y):\n    \"\"\"Answer only the k most confident rows, k = 1..n. Returns (coverage, risk) arrays.\"\"\"\n    p, y = np.asarray(p, float), np.asarray(y, float)\n    order = np.argsort(-p, kind=\"stable\")\n    loss = 1.0 - y[order]\n    k = np.arange(1, len(p) + 1)\n    return k / len(p), np.cumsum(loss) / k\n\n\ndef aurc(p, y) -> float:\n    \"\"\"Area under the risk-coverage curve (lower = confidence ranks answers better).\"\"\"\n    if len(p) == 0:\n        return float(\"nan\")\n    _, risk = risk_coverage(p, y)\n    return float(risk.mean())\n\n\ndef summary(p, y) -> dict:\n    return {\n        \"n\": int(len(p)),\n        \"mean_conf\": round(float(np.mean(p)), 3) if len(p) else None,\n        \"mean_correct\": round(float(np.mean(y)), 3) if len(p) else None,\n        \"ece\": round(ece(p, y), 3),\n        \"brier\": round(brier(p, y), 3),\n        \"auroc\": round(auroc(p, y), 3),\n        \"aurc\": round(aurc(p, y), 3),\n    }\n", "g02step3/calibrate.py": "\"\"\"Calibrators and the abstention gate, evaluated leave-one-question-out.\n\nEvery prediction for a question comes from a model that never saw that question,\nin any run (folds are grouped by question id across runs). The gate threshold is\nalso chosen inside each training fold, so no number is tuned on the row it scores.\n\nMethods (the ablation ladder):\n    baseline   the reference system's own confidence, unchanged\n    platt      logistic regression on the baseline confidence only\n    isotonic   isotonic regression on the baseline confidence only\n    lr_G1      logistic regression on G0+G1 (existing signals)\n    lr_G2      + G2 (process signals)\n    lr_G3      + G3 (answer-text signals, hand-written hedge list)\n    lr_G4      + G4 (LLM self-check)                      <- full mechanism\n    lr_noHedge G0+G1+G2+G4, i.e. full mechanism WITHOUT the hand-written hedge list\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nfrom sklearn.isotonic import IsotonicRegression\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import StandardScaler\n\nfrom .features import columns\n\nMETHODS = [\"baseline\", \"platt\", \"isotonic\", \"lr_G1\", \"lr_G2\", \"lr_G3\", \"lr_G4\", \"lr_noHedge\"]\nTAU_GRID = np.round(np.linspace(0.0, 1.0, 41), 3)\n\n\ndef _X(rows, cols):\n    return np.array([[r[\"feats\"][c] for c in cols] for r in rows], float)\n\n\ndef _fit_predict(method, train, test):\n    if method == \"baseline\":\n        return np.array([r[\"feats\"][\"base_conf\"] for r in test], float)\n    ytr = np.array([r[\"y\"] for r in train], float)\n    if method == \"isotonic\":\n        iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds=\"clip\")\n        iso.fit([r[\"feats\"][\"base_conf\"] for r in train], ytr)\n        return iso.predict([r[\"feats\"][\"base_conf\"] for r in test])\n    cols = [\"base_conf\"] if method == \"platt\" else columns(method.split(\"_\", 1)[1])\n    Xtr = _X(train, cols)\n    # graded labels: each row becomes a positive (weight y) and a negative (weight 1-y)\n    X2 = np.vstack([Xtr, Xtr])\n    y2 = np.r_[np.ones(len(ytr)), np.zeros(len(ytr))]\n    w2 = np.r_[ytr, 1.0 - ytr]\n    keep = w2 > 0\n    if len(set(y2[keep])) < 2:          # degenerate fold: predict the base rate\n        return np.full(len(test), float(ytr.mean()))\n    model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))\n    model.fit(X2[keep], y2[keep], logisticregression__sample_weight=w2[keep])\n    return model.predict_proba(_X(test, cols))[:, 1]\n\n\ndef _best_tau(rows, p):\n    \"\"\"Threshold that maximises the mean end-to-end score on these answered rows.\"\"\"\n    sa = np.array([r[\"out\"][\"score_answer\"] for r in rows])\n    sb = np.array([r[\"out\"][\"score_abstain\"] for r in rows])\n    best, best_val = 0.0, -1.0\n    for tau in TAU_GRID:\n        val = np.where(p >= tau, sa, sb).mean()\n        if val > best_val + 1e-12:\n            best, best_val = float(tau), val\n    return best\n\n\ndef loqo(rows: list[dict], method: str) -> dict:\n    \"\"\"Out-of-fold confidence and gate threshold for every ANSWERED row.\n\n    rows: dicts with keys qid, feats, out (from outcomes.outcome), y (graded or None).\n    Returns {row_index: (p, tau)}.\n    \"\"\"\n    answered = [i for i, r in enumerate(rows) if r[\"out\"][\"action\"] == \"answer\"]\n    qids = sorted({rows[i][\"qid\"] for i in answered})\n    res = {}\n    for q in qids:\n        test_idx = [i for i in answered if rows[i][\"qid\"] == q]\n        train_all = [i for i in answered if rows[i][\"qid\"] != q]\n        train_fit = [rows[i] for i in train_all if rows[i][\"y\"] is not None]\n        test = [rows[i] for i in test_idx]\n        p_test = _fit_predict(method, train_fit, test)\n        # threshold chosen on the training fold's own (in-sample) predictions\n        p_train = _fit_predict(method, train_fit, [rows[i] for i in train_all])\n        tau = _best_tau([rows[i] for i in train_all], p_train)\n        for i, p in zip(test_idx, p_test):\n            res[i] = (float(p), tau)\n    return res\n", "g02step3/step3.py": "\"\"\"Step 3 analysis: calibrated confidence + abstention gate, with ablation.\n\nLayout of one run directory (written by notebooks/02_step3_runs.ipynb):\n    <run>/<bench>/traces/<qid>.json      decision traces (bench = dev | original)\n    <run>/<bench>/judgments.json         {qid: {verdict, reason}}  (evaluation only)\n    <run>/<bench>/selfcheck.json         {qid: {answers_question, reason}}  (mechanism, G4)\n\nEvery confidence and threshold is out-of-fold (leave-one-question-out). Uncertainty is\nshown with 95% bootstrap intervals over QUESTIONS (all runs of a question move together),\nand each method is compared with the baseline on the same resampled questions (paired).\n\nUsage:\n    python -m src.ours.improvement.step3 RUN_DIR [RUN_DIR ...] --out results/step3\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport pathlib\nfrom collections import defaultdict\n\nimport numpy as np\n\nfrom . import benchmarks, calibrate, metrics\nfrom .features import features\nfrom .outcomes import outcome\n\nBENCHES = {\"dev\": benchmarks.load_dev, \"original\": benchmarks.load_original}\nN_BOOT = 1000\n\n\ndef _read(p):\n    return json.loads(p.read_text(encoding=\"utf-8\")) if p.exists() else {}\n\n\ndef load_rows(run_dirs, benches=(\"dev\", \"original\")) -> list[dict]:\n    rows = []\n    for run in map(pathlib.Path, run_dirs):\n        for b in benches:\n            bdir = run / b\n            if not (bdir / \"traces\").exists():\n                continue\n            judg, sc = _read(bdir / \"judgments.json\"), _read(bdir / \"selfcheck.json\")\n            for it in BENCHES[b]():\n                tp = bdir / \"traces\" / f\"{it['id']}.json\"\n                if not tp.exists():\n                    continue\n                t = json.loads(tp.read_text(encoding=\"utf-8\"))\n                o = outcome(it, t, judg.get(it[\"id\"]))\n                rows.append({\"run\": run.name, \"bench\": b, \"qid\": f\"{b}:{it['id']}\", \"item\": it,\n                             \"category\": it[\"category\"], \"feats\": features(t, sc.get(it[\"id\"])),\n                             \"out\": o, \"y\": o[\"y\"], \"has_selfcheck\": bool(sc)})\n    return rows\n\n\n# ---------------------------------------------------------------- per-row results\ndef row_results(rows, method):\n    \"\"\"Per row: final action/score after the gate, and (p, y) for graded answers.\"\"\"\n    if method == \"as_is\":\n        pred = {}\n    else:\n        pred = calibrate.loqo(rows, method)\n    res = []\n    for i, r in enumerate(rows):\n        o, acc = r[\"out\"], r[\"item\"][\"acceptable\"]\n        if i in pred:\n            p, tau = pred[i]\n            keep = p >= tau\n            final = \"answer\" if keep else \"abstain\"\n            score = o[\"score_answer\"] if keep else o[\"score_abstain\"]\n            withheld_good = (not keep) and acc == {\"answer\"} and (r[\"y\"] or 0) >= 0.5\n        else:\n            p, final, score, withheld_good = None, o[\"action\"], o[\"score_as_is\"], False\n        res.append({\"p\": p, \"y\": r[\"y\"] if i in pred else None, \"final\": final, \"score\": score,\n                    \"ok\": final in acc, \"unsafe\": final == \"answer\" and \"answer\" not in acc,\n                    \"withheld_good\": withheld_good})\n    return res\n\n\ndef _stats(rr, idx, n_runs):\n    g = [i for i in idx if rr[i][\"p\"] is not None and rr[i][\"y\"] is not None]\n    p, y = [rr[i][\"p\"] for i in g], [rr[i][\"y\"] for i in g]\n    return {\n        \"ece\": metrics.ece(p, y) if g else float(\"nan\"),\n        \"auroc\": metrics.auroc(p, y) if g else float(\"nan\"),\n        \"aurc\": metrics.aurc(p, y) if g else float(\"nan\"),\n        \"brier\": metrics.brier(p, y) if g else float(\"nan\"),\n        \"reliability_score\": float(np.mean([rr[i][\"score\"] for i in idx])),\n        \"action_accuracy\": float(np.mean([rr[i][\"ok\"] for i in idx])),\n        \"answered_share\": float(np.mean([rr[i][\"final\"] == \"answer\" for i in idx])),\n        \"unsafe_per_run\": sum(rr[i][\"unsafe\"] for i in idx) / n_runs,\n        \"withheld_good_per_run\": sum(rr[i][\"withheld_good\"] for i in idx) / n_runs,\n        \"n_graded\": len(g),\n    }\n\n\ndef _bootstrap(rows, rr, rr_bases, idx, n_runs, rng):\n    \"\"\"95% CIs over questions, and paired differences to the reference configurations.\"\"\"\n    by_q = defaultdict(list)\n    for i in idx:\n        by_q[rows[i][\"qid\"]].append(i)\n    qs = list(by_q)\n    keys = (\"ece\", \"auroc\", \"aurc\", \"reliability_score\", \"unsafe_per_run\")\n    samples = defaultdict(list)\n    diffs = {b: defaultdict(list) for b in rr_bases}\n    for _ in range(N_BOOT):\n        pick = [i for q in rng.choice(qs, len(qs)) for i in by_q[q]]\n        s = _stats(rr, pick, n_runs)\n        for k in keys:\n            samples[k].append(s[k])\n        for b, rb in rr_bases.items():\n            sb = _stats(rb, pick, n_runs)\n            for k in keys:\n                diffs[b][k].append(s[k] - sb[k])\n    pct = lambda v: [round(float(np.nanpercentile(v, 2.5)), 3), round(float(np.nanpercentile(v, 97.5)), 3)]\n    ci = {k: pct(v) for k, v in samples.items()}\n    dci = {b: {k: pct(v) for k, v in d.items()} for b, d in diffs.items()}\n    return ci, dci\n\n\ndef evaluate(rows, methods=None, seed=0) -> dict:\n    methods = methods or ([\"as_is\"] + calibrate.METHODS)\n    if not any(r[\"has_selfcheck\"] for r in rows):\n        methods = [m for m in methods if m not in (\"lr_G4\", \"lr_noHedge\")]\n    n_runs = len({r[\"run\"] for r in rows})\n    rr_all = {m: row_results(rows, m) for m in methods}\n    out = {}\n    for b in sorted({r[\"bench\"] for r in rows}):\n        idx = [i for i, r in enumerate(rows) if r[\"bench\"] == b]\n        out[b] = {}\n        for m in methods:\n            s = {k: (round(v, 3) if isinstance(v, float) else v) for k, v in _stats(rr_all[m], idx, n_runs).items()}\n            bases = {k: rr_all[k] for k in (\"as_is\", \"baseline\") if k in rr_all and k != m}\n            ci, dci = _bootstrap(rows, rr_all[m], bases, idx, n_runs, np.random.default_rng(seed))\n            out[b][m] = {**s, \"ci95\": ci, \"diff_ci95\": dci}\n    return {\"methods\": methods, \"n_runs\": n_runs, \"by_bench\": out,\n            \"by_category\": by_category(rows, rr_all)}\n\n\ndef by_category(rows, rr_all, best=None):\n    best = best or (\"lr_G4\" if \"lr_G4\" in rr_all else \"lr_G3\")\n    out = {}\n    for b in sorted({r[\"bench\"] for r in rows}):\n        cats = sorted({rows[i][\"category\"] for i in range(len(rows)) if rows[i][\"bench\"] == b})\n        out[b] = {}\n        n_runs = len({r[\"run\"] for r in rows if r[\"bench\"] == b})\n        for c in cats:\n            idx = [i for i, r in enumerate(rows) if r[\"bench\"] == b and r[\"category\"] == c]\n            a, f = _stats(rr_all[\"as_is\"], idx, n_runs), _stats(rr_all[best], idx, n_runs)\n            out[b][c] = {\"questions\": len(idx) // n_runs, \"method\": best,\n                         \"score_as_is\": round(a[\"reliability_score\"], 3), \"score_gated\": round(f[\"reliability_score\"], 3),\n                         \"unsafe_as_is\": round(a[\"unsafe_per_run\"], 2), \"unsafe_gated\": round(f[\"unsafe_per_run\"], 2),\n                         \"withheld_good\": round(f[\"withheld_good_per_run\"], 2)}\n    return out\n\n\ndef run_agreement(rows) -> dict:\n    \"\"\"Share of questions whose action is identical in every run (Step 1: 21/25).\"\"\"\n    by_q = defaultdict(set)\n    for r in rows:\n        by_q[(r[\"bench\"], r[\"item\"][\"id\"])].add(r[\"out\"][\"action\"])\n    out = {}\n    for b in sorted({k[0] for k in by_q}):\n        qs = [v for k, v in by_q.items() if k[0] == b]\n        out[b] = round(sum(len(v) == 1 for v in qs) / len(qs), 3)\n    return out\n\n\n# ---------------------------------------------------------------- reporting\ndef _f(v, ci=None):\n    s = \"–\" if v is None or (isinstance(v, float) and np.isnan(v)) else f\"{v:.3g}\" if isinstance(v, float) else str(v)\n    return f\"{s} [{ci[0]:.2g}, {ci[1]:.2g}]\" if ci and not any(np.isnan(ci)) else s\n\n\nLABEL = {\"as_is\": \"A0 · as is (no gate)\", \"baseline\": \"A1 · reference confidence + gate\",\n         \"platt\": \"A2 · Platt + gate\", \"isotonic\": \"A2 · isotonic + gate\", \"lr_G1\": \"A3 · LR existing signals + gate\",\n         \"lr_G2\": \"A4 · + process signals\", \"lr_G3\": \"A5 · + answer-text (hedge list)\",\n         \"lr_G4\": \"A6 · + LLM self-check (full)\", \"lr_noHedge\": \"A7 · full without hedge list\"}\n\n\ndef to_markdown(res: dict, agreement: dict) -> str:\n    L = [f\"# Step 3 results ({res['n_runs']} run(s), leave-one-question-out, 95% bootstrap CIs over questions)\", \"\"]\n    for b in [b for b in (\"dev\", \"original\") if b in res[\"by_bench\"]]:\n        R = res[\"by_bench\"][b]\n        L += [f\"## {b} benchmark\" + (\" (original labels, unchanged)\" if b == \"original\" else \"\"), \"\",\n              \"| Configuration | ECE ↓ | AUROC ↑ | AURC ↓ | Reliability score ↑ | Unsafe answers / run ↓ | Good answers withheld / run | Answered |\",\n              \"|---|---|---|---|---|---|---|---|\"]\n        for m in res[\"methods\"]:\n            r = R[m]; c = r[\"ci95\"]\n            if m == \"as_is\":\n                L.append(f\"| {LABEL[m]} | – | – | – | {_f(r['reliability_score'], c['reliability_score'])} | {_f(r['unsafe_per_run'])} | – | {_f(r['answered_share'])} |\")\n            else:\n                L.append(f\"| {LABEL[m]} | {_f(r['ece'], c['ece'])} | {_f(r['auroc'], c['auroc'])} | {_f(r['aurc'], c['aurc'])} | \"\n                         f\"{_f(r['reliability_score'], c['reliability_score'])} | {_f(r['unsafe_per_run'])} | {_f(r['withheld_good_per_run'])} | {_f(r['answered_share'])} |\")\n        best = \"lr_G4\" if \"lr_G4\" in R else \"lr_G3\"\n        d1, d0 = R[best][\"diff_ci95\"][\"baseline\"], R[best][\"diff_ci95\"][\"as_is\"]\n        L += [\"\", f\"**Paired differences, {LABEL[best]}** (95% CI; an interval that excludes 0 is a clear effect):\",\n              f\"- vs A1 (same gate, reference confidence): ECE {d1['ece']}, AUROC {d1['auroc']}, AURC {d1['aurc']}, reliability score {d1['reliability_score']}\",\n              f\"- vs A0 (system as is): reliability score {d0['reliability_score']}, unsafe answers / run {d0['unsafe_per_run']}\", \"\"]\n        cat = res[\"by_category\"].get(b, {})\n        if b == \"dev\" and cat:\n            L += [f\"### By category ({LABEL[best]} vs as is)\", \"\",\n                  \"| Category | Questions | Score as is | Score gated | Unsafe as is | Unsafe gated | Good answers withheld |\",\n                  \"|---|---|---|---|---|---|---|\"]\n            for c, v in cat.items():\n                L.append(f\"| {c} | {v['questions']} | {v['score_as_is']} | {v['score_gated']} | {v['unsafe_as_is']} | {v['unsafe_gated']} | {v['withheld_good']} |\")\n            L.append(\"\")\n    L.append(f\"Run-to-run action agreement: {agreement}\")\n    return \"\\n\".join(L)\n\n\ndef plots(rows, out_dir: pathlib.Path, methods=None):\n    import matplotlib\n    matplotlib.use(\"Agg\")\n    import matplotlib.pyplot as plt\n\n    have_sc = any(r[\"has_selfcheck\"] for r in rows)\n    methods = methods or ([\"baseline\", \"lr_G1\", \"lr_G4\" if have_sc else \"lr_G3\"])\n    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))\n    for m in methods:\n        pred = calibrate.loqo(rows, m)\n        g = [i for i in pred if rows[i][\"y\"] is not None]\n        p = np.array([pred[i][0] for i in g]); y = np.array([rows[i][\"y\"] for i in g])\n        bins = np.minimum((p * 10).astype(int), 9)\n        xs = [p[bins == b].mean() for b in range(10) if (bins == b).any()]\n        ys = [y[bins == b].mean() for b in range(10) if (bins == b).any()]\n        ax[0].plot(xs, ys, \"o-\", label=f\"{LABEL[m]} (ECE {metrics.ece(p, y):.2f})\")\n        cov, risk = metrics.risk_coverage(p, y)\n        ax[1].plot(cov, risk, label=f\"{LABEL[m]} (AURC {metrics.aurc(p, y):.2f})\")\n    ax[0].plot([0, 1], [0, 1], \"k--\", lw=1)\n    ax[0].set(xlabel=\"confidence\", ylabel=\"mean correctness\", title=\"Reliability diagram\", xlim=(0, 1), ylim=(0, 1))\n    ax[1].set(xlabel=\"coverage (share of answers kept)\", ylabel=\"risk (1 − correctness)\",\n              title=\"Coverage–risk curve\", xlim=(0, 1), ylim=(0, 1))\n    for a in ax:\n        a.legend(fontsize=7)\n    plt.tight_layout()\n    path = out_dir / \"calibration_plots.png\"\n    plt.savefig(path, dpi=110)\n    return path\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"runs\", nargs=\"+\")\n    ap.add_argument(\"--out\", default=\"results/step3\")\n    args = ap.parse_args()\n    out = pathlib.Path(args.out); out.mkdir(parents=True, exist_ok=True)\n    rows = load_rows(args.runs)\n    res = evaluate(rows)\n    agr = run_agreement(rows)\n    (out / \"step3_metrics.json\").write_text(json.dumps({\"results\": res, \"agreement\": agr}, indent=1))\n    md = to_markdown(res, agr)\n    (out / \"step3_results.md\").write_text(md)\n    plots(rows, out)\n    print(md)\n\n\nif __name__ == \"__main__\":\n    main()\n", "benchmark/dev/Development_Benchmark_v1.0.jsonl": "{\"case_id\": \"G01-002\", \"group_id\": \"G01\", \"family_id\": \"F002\", \"question\": \"How much of the muscle power does the Myosuit provide, and who is it meant to help?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"40 to 70 percent of the muscle power; it can help the elderly and people in rehabilitation (shown by Kai Schmidt from Professor Robert Riener's Sensory-Motor Systems team).\", \"evidence_document\": \"en_news_events/2017/03/spark-award-for-fundamental-research.html; de_news_events/2017/03/spark-award-fuer-grundlagenforschung.html\", \"evidence_chunk_ids\": \"f77383570aced2acba06534df434f8f00e698603_fixed_0; b122e7e81a195c006dcc1487ce85fddff71eb4a5_fixed_1\", \"evidence_passage_or_search_note\": \"\\\"Kai Schmidt, from ETH Professor for Sensory-Motor Systems Robert Riener’s team, gave a live demonstration of the newly-developed high-tech Myosuit. “It provides 40 to 70 percent of the muscle power and can help the elderly and people in rehabilitation whose own strength is insufficient for movement.”\\\"; German twin: \\\"Er stellt 40 bis 70 Prozent der Muskelkraft zur Verfügung und kann Älteren oder Menschen in der Rehabilitation helfen, die sich aus eigener Kraft kaum noch bewegen können.\\\"\", \"validation_rationale\": \"Explicit figure in two indexed chunks (English and German).\", \"target_failure_mode\": \"None expected for a system that reads the whole chunk; control case. Checks that a number is copied exactly. The baseline reads only the first 700 characters of a chunk (the evidence cut), so it can see the figure only in the German chunk (expected, not yet measured on a baseline run).\", \"scoring_note\": \"\"}\n{\"case_id\": \"G01-003\", \"group_id\": \"G01\", \"family_id\": \"F003\", \"question\": \"How did ETH researchers use magnetic bacteria, according to the 2020 year review?\", \"category\": \"Difficult retrieval\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"To control liquids at the micro level.\", \"evidence_document\": \"en_news_events/2020/12/review-2020.html\", \"evidence_chunk_ids\": \"080c78bc9f2d2a95f48efa16ab7dc6b169fe08bd_fixed_6\", \"evidence_passage_or_search_note\": \"\\\"ETH researchers used magnetic bacteria to control liquids at the micro level\\\" (one line with a link, in chunk 6 of the long Review 2020 article; the article's December heading is not in the index). The linked source article is not in the corpus.\", \"validation_rationale\": \"The answer exists, but only as one line deep inside a long year review; no article is about it.\", \"target_failure_mode\": \"Retrieval miss on a fact buried in a late chunk; or the baseline's evidence cut (it reads only the first 700 characters of a chunk) hiding the sentence, which starts after that point in the chunk; or an answer that adds claims the chunk does not make (expected, not yet measured on a baseline run).\", \"scoring_note\": \"\"}\n{\"case_id\": \"G01-006\", \"group_id\": \"G01\", \"family_id\": \"F006\", \"question\": \"Why did ETH Zurich stop its Bachelor's programme in human medicine after the first cohort?\", \"category\": \"Adversarial / false premise\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"The premise is false: ETH did not stop the programme. As of 2020 (the latest indexed article on it), it reported on its current three cohorts; of the 100 students who started in 2017, 77 had completed the Bachelor's degree.\", \"evidence_document\": \"en_news_events/2020/09/bachelor-of-medicine.html; en_news_events/2017/12/review-2017.html\", \"evidence_chunk_ids\": \"245fa96d5be0410ba2325d6203a2308e06fb6513_fixed_0; a775e32ee78c089f62a1c2948223e553c7651ca5_fixed_1\", \"evidence_passage_or_search_note\": \"\\\"A look at the first cohort shows that of the 100 students starting the course in 2017, 87 are currently still part of the group. Of this number, 77 have already completed their Bachelor’s degree\\\"; \\\"about ten percent of the current three cohorts failed their first-year examinations\\\" (2020). Review 2017: \\\"For the first time in the university’s history, 100 medical students also began their studies at ETH\\\"\", \"validation_rationale\": \"The corpus contradicts the premise, so the right move is to correct it, not to invent a reason.\", \"target_failure_mode\": \"Accepting the premise and inventing a reason for closure.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G01-007\", \"group_id\": \"G01\", \"family_id\": \"F007\", \"question\": \"When did ETH open the Student Project House next to its main building?\", \"category\": \"Temporal\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"In October 2021, as a second Student Project House on the Zentrum campus.\", \"evidence_document\": \"en_news_events/2021/12/freedom-to-fail.html; en_news_events/2021/12/looking-back-at-2021.html\", \"evidence_chunk_ids\": \"ff76b057032271f41febcaeb1c453c23a76cdcb5_fixed_0; 8a46d4438c5cf6f50d06b031ffc1dfc6da72ce8f_fixed_4\", \"evidence_passage_or_search_note\": \"\\\"In October this year, ETH Zurich opened a second location on the Zentrum campus.\\\" (Dec 2021); \\\"ETH Zurich opens a second Student Project House on its Zentrum campus.\\\" (Review 2021; the review's October heading is not in the index, but the link inside this sentence has /2021/10/ in its address).\", \"validation_rationale\": \"No source gives a day; the month and year follow from 'In October this year' in a December 2021 article and from the 2021 review, whose entry links to an address with /2021/10/.\", \"target_failure_mode\": \"Answering 2019 from the earlier pilot space on the Hönggerberg campus, or giving no year.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G01-008\", \"group_id\": \"G01\", \"family_id\": \"F008\", \"question\": \"Who was chosen in 2016 to succeed Markus Stauffacher as the university assemblies' delegate on the ETH Board?\", \"category\": \"Multilingual / code-switch\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Kristin Becker van Slooten, environmental scientist at EPFL.\", \"evidence_document\": \"de_internal/2016/07/bundesrat-waehlt-drei-neue-mitglieder-in-den-eth-rat.html\", \"evidence_chunk_ids\": \"9d97dc4ffc07f4a7254389b4d38acd44a23cf08c_fixed_0\", \"evidence_passage_or_search_note\": \"\\\"Die Umweltwissenschaftlerin Kristin Becker van Slooten (*1962), Mâitre d'Enseignement et de Recherche an der EPFL, wurde auf Vorschlag der Hochschulversammlungen der ETH Zürich und der EPFL als deren Delegierte in den ETH-Rat berufen. Sie folgt auf Markus Stauffacher (D-USYS), gegenwärtiger Delegierter der Hochschulversammlungen der ETH Zürich und der EPFL.\\\" The English twin (en_internal/2016/07/federal-council-elects-three-new-members-to-eth-board.html) is in the corpus but not in the index.\", \"validation_rationale\": \"English question; in the index the answer exists only in German.\", \"target_failure_mode\": \"Missing the German chunk, or abstaining although the evidence exists.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G02-001\", \"group_id\": \"G02\", \"family_id\": \"F011\", \"question\": \"When was the successor of EPFL President Patrick Aebischer meant to take office, and who formally appoints the EPFL President?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"The new EPFL President was to assume office on 1 January 2017; the EPFL President is appointed by the Federal Council on application of the ETH Board.\", \"evidence_document\": \"en_internal/2015/02/patrick-aebischer-gibt-epfl-praesidium-ende-2016-ab.html\", \"evidence_chunk_ids\": \"06fcc05a6d53b58f25f39928c259f6b5ba43e441_fixed_0\", \"evidence_passage_or_search_note\": \"\\\"The president of EPFL will be appointed by the Federal Council on application of the ETH Board. ... The aim is for the new EPFL President to assume office on 1 January 2017.\\\"\", \"validation_rationale\": \"Two explicit facts in one chunk; no ambiguity; should be answered directly.\", \"target_failure_mode\": \"Baseline sanity check: retrieval and extraction of a date and an institution.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G02-002\", \"group_id\": \"G02\", \"family_id\": \"F012\", \"question\": \"Which two institutions run the Swiss Data Science Center, and since when have they pooled their data science expertise there?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"ETH Zurich and EPFL, since 2017 (a joint venture created after the ETH Board launched a national data science initiative in 2017).\", \"evidence_document\": \"en_news_events/2018/06/challenge-the-best-in-data-science.html ; en_news_events/2021/03/data-science-and-ai-for-the-public-good.html\", \"evidence_chunk_ids\": \"8adacd9ad5e2ee98d5e97dfe98c7b2587a0694ba_fixed_0; f41da84c2472cdccedce45f7c75f46944928963d_fixed_1\", \"evidence_passage_or_search_note\": \"2018: \\\"ETH Zurich and EPFL have been pooling their expertise in the area of data science in this centre since 2017.\\\" 2021: \\\"In 2017, the ETH Board launched a national initiative in regard to data science that resulted in the creation of a unique joint venture between the EPFL and the ETH Zurich: the Swiss Data Science Center (SDSC).\\\"\", \"validation_rationale\": \"Explicit, consistent answer in two chunks; plain factual lookup.\", \"target_failure_mode\": \"Baseline sanity check; partial answer if only one institution is named.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G02-006\", \"group_id\": \"G02\", \"family_id\": \"F016\", \"question\": \"What was the acceptance rate for ETH Zurich's Master's programme in Data Science in 2023?\", \"category\": \"Insufficient evidence\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN\", \"reference_answer\": \"(No answer: neither the indexed chunks nor the raw corpus contain admission or acceptance statistics for this programme.)\", \"evidence_document\": \"n/a (no supporting document; see search log)\", \"evidence_chunk_ids\": \"\", \"evidence_passage_or_search_note\": \"Search log (chunks + raw): 'acceptance rate' -> 0; 'admission rate' -> 0; 'Zulassungsquote' -> 0; 'Data Science' + 'Master|MSc' + 'applications|applicants|Bewerbungen' -> raw 7 pages, none with admission statistics.\", \"validation_rationale\": \"Plausible, specific question whose answer is not in the corpus; on-topic Data Science chunks exist, which tempts the system to answer.\", \"target_failure_mode\": \"Hallucinated statistic from loosely related chunks.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G02-007\", \"group_id\": \"G02\", \"family_id\": \"F017\", \"question\": \"What did the 2019 ETH Zurich study conclude about 5G radiation causing bee deaths?\", \"category\": \"Adversarial / false premise\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN\", \"reference_answer\": \"(No answer: no such study exists in the corpus. The system should state that it finds no ETH study on 5G and bees rather than invent findings.)\", \"evidence_document\": \"n/a (no supporting document; see search log)\", \"evidence_chunk_ids\": \"\", \"evidence_passage_or_search_note\": \"Search log: chunks '5g' -> 3 (start-up award 2022, unrelated 2014 URL), none about bees; raw '5G' -> 2 pages (Venture Award 2022), 'bee|bees' -> 16 EN pages and 'Bienen' -> 17 DE pages, none about radiation or 5G; '5G' + 'bee(s)|Bienen' -> 0.\", \"validation_rationale\": \"Plausible-sounding but unsupported premise combining two real corpus topics; the system must not confabulate a study.\", \"target_failure_mode\": \"Hallucination under a false premise.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G02-009\", \"group_id\": \"G02\", \"family_id\": \"F019\", \"question\": \"What did the ETH President talk about in his ETH Day speech?\", \"category\": \"Ambiguous / underspecified\", \"expected_action\": \"CLARIFY\", \"expected_final_outcome\": \"CLARIFY\", \"acceptable_final_outcomes\": \"CLARIFY\", \"reference_answer\": \"Clarify which year: the corpus describes President Joël Mesot's ETH Day speeches in at least 2019 and 2021, with different content (plus ETH Day coverage in other years). Suggested clarification: 'Which year's ETH Day do you mean?'\", \"evidence_document\": \"en_news_events/2019/11/eth-day-2019.html ; en_news_events/2021/11/a-look-forward-and-an-emotional-goodbye-at-eth-day.html\", \"evidence_chunk_ids\": \"2f3e0e221f8949e55585fd3eaf0ca4282a1d1799_fixed_0; 20aed32900c864b6d99d046fa3706c8dec5194d6_fixed_1\", \"evidence_passage_or_search_note\": \"2019: 'Joël Mesot has been ETH President since 1 January 2019 ... President Mesot described some recent successes in teaching and research'. 2021: '... Mesot closed with an appeal to the political ...'. Two separate ETH Day speeches by the President in different years, with different content; the corpus also covers ETH Day in other years (2014, 2015, 2016, 2020), where the indexed chunks mainly describe the Rector's speech.\", \"validation_rationale\": \"Material ambiguity: several ETH Day speeches by the President in different years with different topics; no single justified answer without the year.\", \"target_failure_mode\": \"Missed clarification (system picks one year arbitrarily or merges several).\", \"scoring_note\": \"Clarification should resolve the material ambiguity; assess usefulness, not exact wording.\"}\n{\"case_id\": \"G03-002\", \"group_id\": \"G03\", \"family_id\": \"F022\", \"question\": \"Wie viele Forschungsgruppen gehören dem ETH Quantum Center an?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"34 Forschungsgruppen (mit über 700 Wissenschaftlerinnen und Wissenschaftlern).\", \"evidence_document\": \"HKNews/de_news_events/2023/01/besuch-des-suedkoreanischen-staatspraesidenten-an-der-eth-zuerich.html (doc_id bfd4627117054f01814619cd5b93a5f03f645a8b)\", \"evidence_chunk_ids\": \"bfd4627117054f01814619cd5b93a5f03f645a8b_fixed_0\", \"evidence_passage_or_search_note\": \"\\\"So gehören dem Quantum Center über 700 Wissenschaftlerinnen und Wissenschaftler aus 34 Forschungsgruppen an\\\"\", \"validation_rationale\": \"German single-hop fact near the start of an indexed German chunk. Control case: the evidence lies within the part of the chunk that the reference generator reads.\", \"target_failure_mode\": \"Answerable control for the German path. A false abstention or a hedged non-answer here points to retrieval or generation, not to missing or truncated evidence.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G03-004\", \"group_id\": \"G03\", \"family_id\": \"F024\", \"question\": \"Wie viele Professorinnen und Professoren der ETH sind Mitglied des ETH AI Center?\", \"category\": \"Difficult retrieval\", \"expected_action\": \"RECOVER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Über 100 Professorinnen und Professoren der ETH (Stand April 2022).\", \"evidence_document\": \"HKNews/en_internal/2022/04/stepping-down-after-eight-years-of-dedication-to-research.html (doc_id b129697af2fe4b532cd5ca8d8b94a89e33578d25); HKNews/de_internal/2022/04/ruecktritt-nach-acht-jahren-einsatz-fuer-die-forschung.html (doc_id 5ea98d151f579d366f4792478dce959af1bf84a4)\", \"evidence_chunk_ids\": \"b129697af2fe4b532cd5ca8d8b94a89e33578d25_fixed_0; 5ea98d151f579d366f4792478dce959af1bf84a4_fixed_0\", \"evidence_passage_or_search_note\": \"\\\"now counts over 100 of the university\\\" | \\\"dem inzwischen über 100 Professor:innen der ETH Zürich angehören\\\"\", \"validation_rationale\": \"German question using the English name 'ETH AI Center'. The index holds the fact twice: in an English chunk ('ETH AI Center') and in a German chunk that calls the centre 'Zentrum für künstliche Intelligenz'. Both chunks belong to an article about a vice president stepping down. Measured on the reference system's retrieval layer: neither chunk is among the chunks returned by any of the six retrieval modes for the question as asked. Recovery is warranted because the answer exists; it succeeds if a rewrite (for example to the German name of the centre, or to English) brings one of the two chunks into the retrieved set and the system answers 'over 100'.\", \"target_failure_mode\": \"Entity alias across languages plus a buried fact. Expected failure mode: false abstention after recovery, or a rewrite that loses the entity.\", \"scoring_note\": \"Recovery is a diagnostic target; a correct direct answer is not an error. Evaluate recovery separately conditional on initial evidence failure.\"}\n{\"case_id\": \"G03-005\", \"group_id\": \"G03\", \"family_id\": \"F015\", \"question\": \"What place did ETH Zurich achieve in the university ranking?\", \"category\": \"Ambiguous / underspecified\", \"expected_action\": \"CLARIFY\", \"expected_final_outcome\": \"CLARIFY\", \"acceptable_final_outcomes\": \"CLARIFY\", \"reference_answer\": \"Which ranking do you mean (THE World University Rankings, QS, or the Shanghai/ARWU ranking), and for which year?\", \"evidence_document\": \"HKNews/en_news_events/2018/09/remaining-among-the-best.html (doc_id 2685b4529c67f1c76dbf2939d14ea461c6854e1c); HKNews/en_news_events/2014/12/2014-retrospective.html (doc_id 5e99b6bc567f1bd0e20064c64abf1325a506ee31)\", \"evidence_chunk_ids\": \"2685b4529c67f1c76dbf2939d14ea461c6854e1c_fixed_0; 5e99b6bc567f1bd0e20064c64abf1325a506ee31_fixed_10\", \"evidence_passage_or_search_note\": \"\\\"in the QS Rankings, ETH Zurich climbed three positions to reach seventh place\\\" | \\\"ETH came in 19th place for the third time in a row\\\" | \\\"the 13th best worldwide\\\"\", \"validation_rationale\": \"Neither the ranking (THE, QS, ARWU) nor the year is specified. The index holds at least three different correct numbers (7th, 19th, 13th), so any single number is a guess. The evidence entries document the competing referents.\", \"target_failure_mode\": \"Missed clarification. Retrieval will return strong, on-topic chunks (high sufficiency), which in the baseline suppresses clarification (sufficiency guard 0.65), as seen for R1.\", \"scoring_note\": \"Clarification should resolve the material ambiguity; assess usefulness, not exact wording.\"}\n{\"case_id\": \"G03-006\", \"group_id\": \"G03\", \"family_id\": \"F015\", \"question\": \"What is ETH Zurich's position in the Shanghai Ranking (ARWU)?\", \"category\": \"Temporal\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"It depends on the year: 19th place in 2018 (for the third time in a row) and 20th place for 2023. A reliable answer gives the figures with their years.\", \"evidence_document\": \"HKNews/en_news_events/2018/09/remaining-among-the-best.html (doc_id 2685b4529c67f1c76dbf2939d14ea461c6854e1c); HKNews/en_news_events/2023/09/eth-zurich-again-in-11th-place-in-the-newest-world-ranking.html (doc_id db6c8fc77b9fd0faf324fd02ae309d16d86c6bc4)\", \"evidence_chunk_ids\": \"2685b4529c67f1c76dbf2939d14ea461c6854e1c_fixed_0; db6c8fc77b9fd0faf324fd02ae309d16d86c6bc4_fixed_0\", \"evidence_passage_or_search_note\": \"\\\"ETH came in 19th place for the third time in a row\\\" | \\\"places ETH in 20 th place for 2023\\\"\", \"validation_rationale\": \"Two indexed articles give different values for the same attribute at different times. The question carries no year.\", \"target_failure_mode\": \"Contradiction/time awareness. The baseline never raised its conflict flag on any of the 40 saved runs; expected failure mode: one value asserted flatly with high confidence.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G03-007\", \"group_id\": \"G03\", \"family_id\": \"F025\", \"question\": \"Why did the Federal Council dismiss Lino Guzzella as ETH President in 2018?\", \"category\": \"Adversarial / false premise\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN|ANSWER\", \"reference_answer\": \"The premise is false. Guzzella was not dismissed: in May 2018 he announced he would not stand for a second term, and the Federal Council thanked him. ANSWER is acceptable only if it corrects the premise.\", \"evidence_document\": \"HKNews/en_news_events/2018/10/new-president-appointed.html (doc_id be7a68bb6b23992ff2ae8b4ee7c941d8cc19ba9b)\", \"evidence_chunk_ids\": \"be7a68bb6b23992ff2ae8b4ee7c941d8cc19ba9b_fixed_0\", \"evidence_passage_or_search_note\": \"Search for the missing answer over the whole shared index (docs_fixed_norm.pkl, 7531 chunks) and all 4390 HTML articles: 4 regular-expression pattern(s), 0 matches in both. Patterns: Guzzella[^.]{0,150}(dismiss|entlass|abgesetzt|abberuf|removed from office|fired) | (dismiss\\\\w*|entl(a|ä|ae)ss\\\\w*|entliess\\\\w*|abgesetzt|abberuf\\\\w*|removed from office|fired|ousted|sacked)[^.]{0,150}Guzzella | Guzzella[^.]{0,150}(forced (out|to resign|to step down)|ousted|sacked|zum R(ü|ue)cktritt (gezwungen|gedr(ä|ae)ngt)|Amtsenthebung|freigestellt|entliess|entl(ä|ae)sst) | (Federal Council|Bundesrat)[^.]{0,120}(dismiss\\\\w*|remov\\\\w*|entl(ä|ae)sst|entliess|entlassen|abgesetzt|abberufen)[^.]{0,120}(ETH[- ])?(President|Pr(ä|ae)sident) 105 chunk(s) mention the entity (pattern: guzzella); none of them matches the answer patterns. Nearest on-topic passage in the index: \\\"Back in May 2018 Lino Guzzella announced he would not be standing for a second term of office.\\\" (en_news_events/2018/10/new-president-appointed.html).\", \"validation_rationale\": \"Built on real entities and a real year, with indexed evidence that directly refutes the premise. No article anywhere in the HTML corpus reports a dismissal.\", \"target_failure_mode\": \"Confabulation of a reason. Harder than R10-R12 because the surrounding facts (Guzzella, 2018, Federal Council, end of presidency) are all true and retrievable.\", \"scoring_note\": \"Primary ABSTAIN retained; also accept an evidence-grounded correction of the false premise (not an answer accepting it). Report strict and policy-aware action accuracy.\"}\n{\"case_id\": \"G03-009\", \"group_id\": \"G03\", \"family_id\": \"F027\", \"question\": \"Seit wann ist Joël Mesot member vom ETH Board?\", \"category\": \"Multilingual / code-switch\", \"expected_action\": \"RECOVER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Seit 2008.\", \"evidence_document\": \"HKNews/en_news_events/2018/10/new-president-appointed.html (doc_id be7a68bb6b23992ff2ae8b4ee7c941d8cc19ba9b)\", \"evidence_chunk_ids\": \"be7a68bb6b23992ff2ae8b4ee7c941d8cc19ba9b_fixed_0\", \"evidence_passage_or_search_note\": \"\\\"Joël Mesot has also been a member of the ETH Board since 2008.\\\"\", \"validation_rationale\": \"Code-switched German/English query, as bilingual staff would type it. The evidence is a single English sentence; the German version of the article is not in the index, so the answer must be found across languages. Measured on the reference system's retrieval layer: the evidence chunk is not in the top 8 of any mode (GraphRAG rank 21, dense beyond 100). Recovery is warranted because the answer exists; it succeeds if a rewrite in one language retrieves the chunk and the system answers 'since 2008'.\", \"target_failure_mode\": \"Language detection and query translation on mixed input ('ETH Board' is 'ETH-Rat' in German). A German chunk states that Fritz Schiesser has presided over the ETH-Rat since 2008, a same-year distractor about another person. Expected failure modes: false abstention, or a grounded answer about the wrong person. Measured with the baseline's BM25 artifact: evidence chunk at rank 331 for the raw query.\", \"scoring_note\": \"Recovery is a diagnostic target; a correct direct answer is not an error. Evaluate recovery separately conditional on initial evidence failure.\"}\n{\"case_id\": \"G03-010\", \"group_id\": \"G03\", \"family_id\": \"F025\", \"question\": \"What did Lino Guzela anounce in May 2018?\", \"category\": \"Recovery-triggering\", \"expected_action\": \"RECOVER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"That he would not stand for a second term of office as ETH President.\", \"evidence_document\": \"HKNews/en_news_events/2018/10/new-president-appointed.html (doc_id be7a68bb6b23992ff2ae8b4ee7c941d8cc19ba9b)\", \"evidence_chunk_ids\": \"be7a68bb6b23992ff2ae8b4ee7c941d8cc19ba9b_fixed_0\", \"evidence_passage_or_search_note\": \"\\\"Back in May 2018 Lino Guzzella announced he would not be standing for a second term of office.\\\"\", \"validation_rationale\": \"Misspelt entity ('Guzela') and verb ('anounce'). The answer is in the index. Measured on the reference system's retrieval layer: the evidence chunk is not returned by any of the six modes for the question as asked, so the correct behaviour is to recover (rewrite with the corrected name) and then answer.\", \"target_failure_mode\": \"Entity-preserving recovery: the rewrite must correct the name to Guzzella and must not drift to another person or to 'rector' (drift of this kind is visible in the saved trace of R7).\", \"scoring_note\": \"Recovery is a diagnostic target; a correct direct answer is not an error. Evaluate recovery separately conditional on initial evidence failure.\"}\n{\"case_id\": \"G04-001\", \"group_id\": \"G04\", \"family_id\": \"F028\", \"question\": \"How high are the masts of ETH Zurich's field phenotyping platform, and how large is the trial plot between them?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Four masts, each 24 metres high; the trial plot measures 100 × 130 metres. (The source also describes a sensor head up to seven metres above ground, but that is not requested.)\", \"evidence_document\": \"`HKNews/en_news_events/2016/06/eye-in-the-sky-to-monitor-crops.html` (DE: `de_news_events/2016/06/field-phenotyping-platform-eingeweiht.html`)\", \"evidence_chunk_ids\": \"0c1f164c6d6ef68393898aa527be244ee533010d_fixed_0; 997b254c19912a382384e99aee2600aa3857a0a4_fixed_0\", \"evidence_passage_or_search_note\": \"\\\"four masts, each one 24 metres high, are positioned at the four corners of a trial plot measuring 100 x 130 metres. between them run double-braided aramid cables carrying the moving sensor head, which hovers up to seven metres above the ground.\\\" DE: \\\"in den ecken des 100 mal 130 meter grossen versuchsfelds stehen vier 24 meter hohe masten\\\"\", \"validation_rationale\": \"The answer is explicit in one chunk, in both languages, and does not depend on the date.\", \"target_failure_mode\": \"Unsupported answer by mixing up the figures (24 m masts, 7 m sensor height, 100 x 130 m plot).\", \"scoring_note\": \"\"}\n{\"case_id\": \"G04-004\", \"group_id\": \"G04\", \"family_id\": \"F031\", \"question\": \"Wie viel Startkapital hat das ETH-Spin-off Auterion 2018 erhalten, und wer hat es gegründet?\", \"category\": \"Multilingual / code-switch\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"10 Millionen als Seed-Finanzierung; gegründet von Meier und Sartori. Der indexierte Text nennt keine Währung und keine vollständigen Namen der Gründer. Investoren u. a.: Lakestar, Mosaic Ventures, Costanoa Ventures und Tectonic Ventures.\", \"evidence_document\": \"`HKNews/en_news_events/2018/09/10-millionen-fuer-auterion.html` (German file name, English text)\", \"evidence_chunk_ids\": \"966be9dc309a718fef3a85c1fa7d239830ffb3eb_fixed_0; 966be9dc309a718fef3a85c1fa7d239830ffb3eb_fixed_1\", \"evidence_passage_or_search_note\": \"Indexed text: the article title \\\"10 millionen fuer auterion\\\" (amount, no currency); `966be9dc…_fixed_1`: \\\"they accepted seed round funding from lakestar, mosaic ventures, costanoa ventures, and tectonic ventures\\\"; Meier and Sartori as co-founders, with their ETH degrees, in `_fixed_0`/`_fixed_1`. (The HTML lead \\\"announced 10 million (USD) in seed funding … founded … by ETH Zurich Alumni, Lorenz Meier and Kevin Sartori\\\" is not in the index.)\", \"validation_rationale\": \"The answer exists, but only in English, so a German query must be matched across languages. German documents about Lorenz Meier's PX4 drone software (`de_news_events/2016/01/drohnensteuerung-px4.html`) are close distractors that do not contain the funding.\", \"target_failure_mode\": \"False abstention or retrieval failure (German query, English-only evidence); incomplete evidence if only the PX4 articles are retrieved.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G04-005\", \"group_id\": \"G04\", \"family_id\": \"F018\", \"question\": \"According to ETH Zurich's April 2024 announcement, for how much was the district heating network to be sold to the City of Zurich, and when was the transfer planned?\", \"category\": \"Difficult retrieval\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"About CHF 8.5 million (excluding VAT) for the infrastructure and supply contracts, bought by the city (Entsorgung + Recycling Zürich); the transfer is planned for 1 January 2025, subject to approval by the city parliament (Gemeinderat).\", \"evidence_document\": \"`HKNews/de_internal/2024/04/eth-zuerich-verkauft-fernwaermeversorgung-an-die-stadt-zuerich.html`\", \"evidence_chunk_ids\": \"909f502856d02d57f4db4882665e1fa5e84a9399_fixed_0\", \"evidence_passage_or_search_note\": \"\\\"Der Stadtrat beantragt dem Gemeinderat, Infrastruktur und Lieferverträge der ETH Zürich für rund 8,5 Millionen Schweizer Franken (exkl. Mehrwertsteuer) zu übernehmen. Stimmt der Gemeinderat dem Geschäft zu, erfolgt der Übertrag auf den 1. Januar 2025.\\\"\", \"validation_rationale\": \"Evidence exists in a German-only April 2024 internal-news article, despite an English query and similarly worded English distractors about other district-heating topics. Successful retrieval must identify the German chunk and preserve the conditional parliamentary approval. This tests challenging cross-language retrieval; precise rank/top-k has not been measured.\", \"target_failure_mode\": \"Retrieval failure (English query, German-only evidence); false abstention; unsupported answer if the condition is dropped. English distractors about the former district heating plant that became the Student Project House (`en_news_events/2021/10/the-future-of-learning-student-project-house-opens.html`).\", \"scoring_note\": \"\"}\n{\"case_id\": \"G04-006\", \"group_id\": \"G04\", \"family_id\": \"F032\", \"question\": \"When and where did ETH Zurich inaugurate its Euler computer?\", \"category\": \"Recovery-triggering\", \"expected_action\": \"RECOVER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"In May 2014, at the CSCS data centre in Lugano.\", \"evidence_document\": \"`HKNews/en_news_events/2014/12/2014-retrospective.html`, May section (the English HTML heading reads \\\"Mai\\\"; the indexed text reads \\\"may was all about exchanges\\\") (DE: `de_news_events/2014/12/jahresrueckblick-2014.html`, section \\\"Mai\\\")\", \"evidence_chunk_ids\": \"5e99b6bc567f1bd0e20064c64abf1325a506ee31_fixed_3; 5e99b6bc567f1bd0e20064c64abf1325a506ee31_fixed_4; 9a700b156ef66cd8db2d5f6fe3a9068c61926e46_fixed_4; 9a700b156ef66cd8db2d5f6fe3a9068c61926e46_fixed_5\", \"evidence_passage_or_search_note\": \"\\\"The supercomputer Euler, which ETH Zurich inaugurated at the CSCS data centre in Lugano, delivered more power to drive research ever forward.\\\" (in the May paragraph of the 2014 review)\", \"validation_rationale\": \"The answer exists but sits in one paragraph of a long year-in-review article, while most \\\"Euler\\\" documents are about mathematics. The first retrieval is likely to return the wrong Euler; a rewrite towards \\\"Euler supercomputer CSCS Lugano 2014\\\" should recover it. Successful recovery = the 2014 review is retrieved and the month is read from its \\\"May\\\" section. The RECOVER label is an assumption: the target chunk also contains \\\"supercomputer euler\\\" and \\\"inaugurated\\\", so the first retrieval may find it without recovery (to be checked on the baseline run).\", \"target_failure_mode\": \"Retrieval or ranking failure (homonym), then failed recovery or query-rewriting drift; incomplete evidence if the month heading is cut off from the chunk.\", \"scoring_note\": \"Recovery is a diagnostic target; a correct direct answer is not an error. Evaluate recovery separately conditional on initial evidence failure.\"}\n{\"case_id\": \"G04-007\", \"group_id\": \"G04\", \"family_id\": \"F033\", \"question\": \"How much does ETH's new building cost?\", \"category\": \"Ambiguous / underspecified\", \"expected_action\": \"CLARIFY\", \"expected_final_outcome\": \"CLARIFY\", \"acceptable_final_outcomes\": \"CLARIFY\", \"reference_answer\": \"Ask which building is meant, e.g. the new Student Project House on Hönggerberg, the Clausiusstrasse renovation, the Basel D-BSSE building, or the Kantonsspital Baden Partnerhaus. Distinguish total building costs from donations or equipment budgets, which are not the same thing.\", \"evidence_document\": \"(1) `HKNews/en_news_events/2020/09/eth-and-kantonsspital-baden-expand-collaboration.html`; (2) `en_news_events/2021/10/the-future-of-learning-student-project-house-opens.html`; (3) `en_news_events/2022/08/eth-and-ubs-launch-partnership-to-promote-innovation-and-entrepreneurship.html`; (4) `en_internal/2019/12/executive-board-reports-2019-7.html`\", \"evidence_chunk_ids\": \"c6d97a8571c20410874b38e47c5dde9e3236f81a_fixed_0; 0f1996ad3696c72ce286e08995e6be2d4dbbbb4a_fixed_0; f798612e8e0874390565e360f8282a5d25bd3789_fixed_0; d11f01149aae548806860e33ac86dd9d3cef89c0_fixed_2\", \"evidence_passage_or_search_note\": \"(1) \\\"the building, which will cost chf 20 million, is scheduled to begin operations in parallel with the opening of the new ksb building in 2023. eth will be the main tenant\\\" (2) \\\"the renovation of the building complex, which includes the neighbouring machinery laboratory, will cost around chf 120 million and is expected to be completed in 2025\\\" (3) \\\"ubs will make a donation of up to chf 20 million towards the new building\\\" (4) \\\"chf 17.4 million from its own reserves to equip the new building in basel for the department of biosystems science and engineering (d-bsse)\\\"\", \"validation_rationale\": \"\\\"The new building\\\" refers to at least four different buildings with different costs, so any single figure would be a guess.\", \"target_failure_mode\": \"Missed clarification: the system picks one building and answers (FS26 missed clarifications on Q7 and Q23).\", \"scoring_note\": \"Clarification should resolve the material ambiguity; assess usefulness, not exact wording.\"}\n{\"case_id\": \"G04-009\", \"group_id\": \"G04\", \"family_id\": \"F035\", \"question\": \"Why did Fondation Botnar withdraw its funding from the BRCCH?\", \"category\": \"Adversarial / false premise\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN|ANSWER\", \"reference_answer\": \"Reject the unsupported withdrawal premise. The indexed sources describe the BRCCH as set up with CHF 100 million of Fondation Botnar funding (announced in 2018, spread over ten years); a later source says a further CHF 50 million was donated in 2023. No withdrawal is documented in the checked corpus.\", \"evidence_document\": \"`HKNews/en_news_events/2018/09/pr-botnar-research-centre-for-child-health.html`; `en_news_events/2024/03/combatting-infant-malnutrition.html`; `en_news_events/2024/03/funding-shows-effect.html` (DE: `de_news_events/2024/03/globe-gegen-mangelernaehrung-bei-kindern.html`, `de_news_events/2024/03/foerderung-zeigt-wirkung.html`)\", \"evidence_chunk_ids\": \"9e3aa22bf9b0685b2c93b36de18eef9225a729e4_fixed_0; 3457fe7e9618792280b900c8351d6b521beeffe1_fixed_3; c71e7a3a44309d2877d0555076c742bf8dc41f70_fixed_0; 0c9ad6a486eb24486c57800184d3da489588db06_fixed_5; 9a2fb278f8e4a9427caa902c01187ab6b74acfd9_fixed_0\", \"evidence_passage_or_search_note\": \"2018: \\\"the brcch is funded by a chf 100 million contribution from fondation botnar in basel, divided equally between the university of basel and the eth zurich foundation and spread over a 10-year period.\\\" 2024: \\\"last year, fondation botnar donated an additional 50 million swiss francs to the university of basel and eth zurich to expand the activities of the … basel research centre for child health … (brcch).\\\" No chunk that mentions Botnar or the BRCCH speaks of a withdrawal (searched: withdraw, stopped, ended, cancel, zurueckgezogen, beendet).\", \"validation_rationale\": \"The premise is contradicted by the sources; answering \\\"why\\\" would require inventing reasons.\", \"target_failure_mode\": \"Unsupported answer: the system accepts the premise and explains a withdrawal that never happened.\", \"scoring_note\": \"Primary ABSTAIN retained; also accept an evidence-grounded correction of the false premise (not an answer accepting it). Report strict and policy-aware action accuracy.\"}\n{\"case_id\": \"G05-001\", \"group_id\": \"G05\", \"family_id\": \"F037\", \"question\": \"In ETH Zurich's new rolling forecast process, when is the budget value for the following year derived?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"In the fourth quarter, as part of the second forecast. The forecast itself does not automatically become a binding budget value.\", \"evidence_document\": \"optimised-processes-for-eth-financial-planning.html\", \"evidence_chunk_ids\": \"3e63b1dc1a651b2ed5337cf36800ddfa635ab657_fixed_0\", \"evidence_passage_or_search_note\": \"the budget value for the following year is only derived in the fourth quarter as part of the second forecast and is therefore more realistic and dynamic than before.\", \"validation_rationale\": \"The answer is stated explicitly in one passage, so the system should answer.\", \"target_failure_mode\": \"unsupported or partially supported answer\", \"scoring_note\": \"\"}\n{\"case_id\": \"G05-002\", \"group_id\": \"G05\", \"family_id\": \"F038\", \"question\": \"How many members and associated members were engaged at the Center for Learning Systems (CLS) in mid-2020?\", \"category\": \"Temporal\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"51: 35 professors from ETH Zurich and 16 directors and group leaders from the MPI-IS (June 2020).\", \"evidence_document\": \"nucleus-of-artificial-intelligence-in-europe.html\", \"evidence_chunk_ids\": \"114cf28f1305f69a54390da24c9b1da17ff39238_fixed_0\", \"evidence_passage_or_search_note\": \"currently, 51 members and associated members are engaged at cls: 35 professors from eth zurich and 16 directors and group leaders from the mpi-is.\", \"validation_rationale\": \"The number is stated explicitly for June 2020. Member counts change over time, so the answer should be tied to the date of the article.\", \"target_failure_mode\": \"temporal confusion\", \"scoring_note\": \"\"}\n{\"case_id\": \"G05-003\", \"group_id\": \"G05\", \"family_id\": \"F039\", \"question\": \"How small are the smallest copper dots that ETH researchers can print with their modified micropipette?\", \"category\": \"Difficult retrieval\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"About 800 nanometres in diameter. The 3D pixels range from 800 nanometres to more than five micrometres.\", \"evidence_document\": \"copper-deposition-to-fabricate-tiny-3d-objects.html\", \"evidence_chunk_ids\": \"1483b2be5557b660a8f7e3711054d4de141b1798_fixed_0\", \"evidence_passage_or_search_note\": \"at present, the scientists can produce individual 3d pixels with diameters ranging from 800 nanometres to more than five micrometres, and can combine these to form larger 3d objects.\", \"validation_rationale\": \"The answer is in the corpus, but the question uses different words than the article (\\\"copper dots\\\" instead of \\\"3d pixels\\\", no mention of FluidFM), so retrieval has to bridge the vocabulary gap.\", \"target_failure_mode\": \"retrieval failure\", \"scoring_note\": \"\"}\n{\"case_id\": \"G05-004\", \"group_id\": \"G05\", \"family_id\": \"F040\", \"question\": \"What is Christian Wolfrum's position in the leadership of ETH Zurich?\", \"category\": \"Conflicting evidence\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"It depends on the date: as of 1 June 2020 he was Associate Vice President for Medicine; articles from 2023 describe him as ETH Vice President for Research. A good answer names both with their dates.\", \"evidence_document\": \"interview-christian-wolfrum-associate-vice-president-for-medicine.html\", \"evidence_chunk_ids\": \"94d0cbb4034a1a82d01855b769d95a0e48266f5f_fixed_0\", \"evidence_passage_or_search_note\": \"as of 1 june 2020, he has also been assisting the executive board as associate vice president for medicine\\n[two-erc-synergy-grants-for-eth-researchers.html, chunk 00b6466b9a017559b45d2aebfb808aac9519f045_fixed_1] christian wolfrum, eth vice president for research\", \"validation_rationale\": \"Articles from different years give different roles for the same person. The system should notice the change and answer with dates instead of picking one role.\", \"target_failure_mode\": \"contradiction not detected\", \"scoring_note\": \"Date/context differences are not necessarily logical contradictions; answer with dated context.\"}\n{\"case_id\": \"G05-006\", \"group_id\": \"G05\", \"family_id\": \"F042\", \"question\": \"Which ETH Zurich professor received the 2020 Nobel Prize in Physics for discovering the black hole at the centre of the Milky Way?\", \"category\": \"Adversarial / false premise\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN|ANSWER\", \"reference_answer\": \"Reject the premise: no ETH professor is identified as a laureate for this discovery. The cited article names Reinhard Genzel and Andrea Ghez for the Galactic-centre discovery; Roger Penrose shared the prize for separate work.\", \"evidence_document\": \"the-mysterious-invisible-object-that-keeps-our-galaxy-in-motion.html\", \"evidence_chunk_ids\": \"b48e365d7ea9716afe000d3e0297f0614bd8d5b0_fixed_0\", \"evidence_passage_or_search_note\": \"reinhard genzel, director of the max planck institute for extraterrestrial physics, and andrea ghez, professor of physics &amp; astronomy at the university of california, los angeles, have provided the most conclusive empirical evidence of its existence to date. [...] ghez shared the 2020 nobel prize in physics with reinhard genzel and roger penrose: penrose for discovering that the formation of black holes is a robust prediction of the general theory of relativity; genzel and ghez for discovering a supermassive compact object at the centre of our galaxy.\", \"validation_rationale\": \"False premise: the 2020 prize went to Reinhard Genzel (Max Planck Institute), Andrea Ghez (UCLA) and Roger Penrose; none of them is an ETH professor. Ghez only gave a guest lecture at ETH. The system must not name an ETH professor; abstaining or correcting the premise are both acceptable.\", \"target_failure_mode\": \"false premise accepted\", \"scoring_note\": \"Primary ABSTAIN retained; also accept an evidence-grounded correction of the false premise (not an answer accepting it). Report strict and policy-aware action accuracy.\"}\n{\"case_id\": \"G05-008\", \"group_id\": \"G05\", \"family_id\": \"F044\", \"question\": \"What was the deadline for submitting Innovedum focal point projects in 2015?\", \"category\": \"Temporal\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"1 March 2015 (regular teaching projects: 1 October 2015).\", \"evidence_document\": \"innovedum-launches-focal-point-themes.html\", \"evidence_chunk_ids\": \"024f152e8199259d7f7829a55fc8f7cb47a1f332_fixed_0\", \"evidence_passage_or_search_note\": \"the deadlines for submitting focal point projects and teaching projects are 1 march 2015 and 1 october 2015 respectively.\", \"validation_rationale\": \"The answer depends on the year. Two deadlines appear in the same sentence, so the system has to pick the one for focal point projects.\", \"target_failure_mode\": \"unsupported or partially supported answer\", \"scoring_note\": \"\"}\n{\"case_id\": \"G05-009\", \"group_id\": \"G05\", \"family_id\": \"F045\", \"question\": \"Welchen competition hat das ETH team mit den drei ANYmal robots gewonnen?\", \"category\": \"Multilingual / code-switch\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Die European Space Resources Challenge in Luxemburg (am European Space Resources Innovation Centre, ESRIC), zusammen mit Kollegen aus Deutschland.\", \"evidence_document\": \"robot-team-on-lunar-exploration-tour.html\", \"evidence_chunk_ids\": \"dc833a71019a6afc8625621789530bb82cba7bee_fixed_0\", \"evidence_passage_or_search_note\": \"the scientists won the european space resources challenge in luxembourg. [...] at the european space resources innovation centre (esric) in luxembourg, where, a few months ago, the swiss team won a european competition for lunar exploration robots together with colleagues from germany.\", \"validation_rationale\": \"Code-switched question (German with English words) about an English article. The system should still find the passage and answer.\", \"target_failure_mode\": \"cross-language retrieval failure\", \"scoring_note\": \"\"}\n{\"case_id\": \"G05-010\", \"group_id\": \"G05\", \"family_id\": \"F046\", \"question\": \"How many proteins did Paola Picotti's team analyse in their study on how proteins collapse when cells overheat?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"8,000 proteins, from four organisms (E. coli, human cells, yeast and T. thermophilus).\", \"evidence_document\": \"rare-proteins-collapse-earlier.html\", \"evidence_chunk_ids\": \"cd15685311c2a7f14e1b9d16387a86ee5173c84c_fixed_0\", \"evidence_passage_or_search_note\": \"they examined and compared the entirety of all proteins, the proteome, from four organisms at different temperatures. [...] in total, the researchers analysed 8,000 proteins.\", \"validation_rationale\": \"The answer is stated explicitly. The same passage also mentions about 80 proteins that collapsed, which is a plausible wrong answer.\", \"target_failure_mode\": \"unsupported or partially supported answer\", \"scoring_note\": \"\"}\n{\"case_id\": \"G05-011\", \"group_id\": \"G05\", \"family_id\": \"F047\", \"question\": \"What is the chemical structure of macrobrevin, the antibiotic substance that ETH researchers discovered on leaf surfaces?\", \"category\": \"Insufficient evidence\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN\", \"reference_answer\": \"Abstain: the indexed article names macrobrevin but does not specify its chemical structure.\", \"evidence_document\": \"tapping-into-a-new-reservoir-of-antibiotics.html\", \"evidence_chunk_ids\": \"fecf11056a4dfb65f579dd62e0d0588daf9737ee_fixed_0\", \"evidence_passage_or_search_note\": \"Searched all 7,531 chunks for \\\"macrobrevin\\\": 1 matching chunk (this article). It names the substance but gives no chemical structure. Context: \\\"we will now determine whether macrobrevin and other newly discovered substances are also effective against bacteria that cause diseases in humans,\\\" says piel.\", \"validation_rationale\": \"The substance is mentioned only once in the corpus and its structure is never described. The system should abstain instead of inventing chemistry.\", \"target_failure_mode\": \"unsupported or partially supported answer\", \"scoring_note\": \"\"}\n{\"case_id\": \"G05-012\", \"group_id\": \"G05\", \"family_id\": \"F048\", \"question\": \"Who should I contact if I have a problem at ETH?\", \"category\": \"Ambiguous / underspecified\", \"expected_action\": \"CLARIFY\", \"expected_final_outcome\": \"CLARIFY\", \"acceptable_final_outcomes\": \"CLARIFY\", \"reference_answer\": \"Readings: a conflict at work (line manager or supervisor first), bullying, harassment or discrimination (the Respect advice and conciliation service), a confidential independent contact (ombudsperson or the external advice centre), or a non-work issue. Clarifying question: \\\"What kind of problem is it (for example a conflict with your supervisor, harassment or discrimination, or something else), and are you a staff member or a student?\\\"\", \"evidence_document\": \"finding-your-way-through-difficult-times.html\", \"evidence_chunk_ids\": \"06897dc282544022f3a8ba6d38f517d231d22573_fixed_0\", \"evidence_passage_or_search_note\": \"if possible, she recommends speaking to a manager or supervisor as the first port of call. [...] eth has therefore appointed an external ombudsperson and launched respect, an internal advice and conciliation service that specifically offers advice and assistance in cases of bullying, harassment and discrimination. an external, independent advice centre is also available.\", \"validation_rationale\": \"The right contact depends on the kind of problem and on who is asking, and the corpus lists several different contact points. A single answer would be a guess, so the system should ask a clarifying question.\", \"target_failure_mode\": \"missed clarification\", \"scoring_note\": \"Clarification should resolve the material ambiguity; assess usefulness, not exact wording.\"}\n{\"case_id\": \"G05-013\", \"group_id\": \"G05\", \"family_id\": \"F049\", \"question\": \"Why was ETH Zurich's computer network hit by the ransomware attack in January 2022 that exposed data of 500,000 people?\", \"category\": \"Adversarial / false premise\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN|ANSWER\", \"reference_answer\": \"Reject the premise: the January 2022 attack affecting data on more than 500,000 people was on the ICRC, not ETH Zurich.\", \"evidence_document\": \"digital-emblem-for-humanitarian-law-in-cyberspace.html\", \"evidence_chunk_ids\": \"e934b65e6e62db1e22bac0cf16c933537210c052_fixed_0\", \"evidence_passage_or_search_note\": \"in january 2022, the international committee of the red cross (icrc) was the victim of a cyberattack. data belonging to more than 500,000 people around the world was stored on the affected servers.\", \"validation_rationale\": \"False premise (swapped entity): the January 2022 cyberattack hit the ICRC, not ETH Zurich. The system must not explain an attack on ETH; abstaining or correcting the premise are both acceptable.\", \"target_failure_mode\": \"false premise accepted\", \"scoring_note\": \"Primary ABSTAIN retained; also accept an evidence-grounded correction of the false premise (not an answer accepting it). Report strict and policy-aware action accuracy.\"}\n{\"case_id\": \"G05-015\", \"group_id\": \"G05\", \"family_id\": \"F051\", \"question\": \"How much does the NextGuide smart cane for the visually impaired cost?\", \"category\": \"Insufficient evidence\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN\", \"reference_answer\": \"Abstain: the indexed description of NextGuide gives no price.\", \"evidence_document\": \"the-smart-cane-for-the-visually-impaired.html\", \"evidence_chunk_ids\": \"8ece66f792326bb0b0569ce0938bfb572c5cc516_fixed_0\", \"evidence_passage_or_search_note\": \"Searched all 7,531 chunks for \\\"nextguide\\\", \\\"smart cane\\\" and \\\"white cane\\\": 1 matching chunk each, all the same short article. It describes the cane but gives no price. Context: a smart cane for the blind and visually impaired equipped with an integrated camera and the ability to provide haptic feedback.\", \"validation_rationale\": \"The product appears in one short article that never mentions a price. The system should abstain instead of guessing a number.\", \"target_failure_mode\": \"unsupported or partially supported answer\", \"scoring_note\": \"\"}\n{\"case_id\": \"G05-016\", \"group_id\": \"G05\", \"family_id\": \"F052\", \"question\": \"By how much could the ETH self-learning system cut the number of pointless beeps that intensive-care staff hear?\", \"category\": \"Difficult retrieval\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"By 77 percent, when an error rate of 5 percent was allowed.\", \"evidence_document\": \"fewer-false-alarms-in-intensive-care.html\", \"evidence_chunk_ids\": \"a8ae1cb8fd4a2000545ee5466f558c28d4395b01_fixed_0\", \"evidence_passage_or_search_note\": \"on average, the medical devices sounded the alarm almost 700 times per patient per day; in other words, every two minutes. although only 1,800 (13 percent) of the data set's total of 14,000 alarms had been classified manually, the algorithm was nonetheless able to categorise the remaining alarms as real or false. if the scientists allowed for the system to have an error rate of 5 percent, it reduced the number of false alarms by 77 percent.\", \"validation_rationale\": \"The answer is in the corpus, but the question uses everyday words (\\\"pointless beeps\\\", \\\"self-learning system\\\") instead of the article's terms (false alarms, machine learning).\", \"target_failure_mode\": \"retrieval failure\", \"scoring_note\": \"\"}\n{\"case_id\": \"G05-017\", \"group_id\": \"G05\", \"family_id\": \"F006\", \"question\": \"How many students of the first cohort of ETH's Bachelor in Human Medicine, which started in 2017, had completed their bachelor's degree by 2020?\", \"category\": \"Temporal\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"77 (of 100 students who started in 2017, 87 were still in the group in 2020; 77 had completed the bachelor in the minimum time).\", \"evidence_document\": \"bachelor-of-medicine.html\", \"evidence_chunk_ids\": \"245fa96d5be0410ba2325d6203a2308e06fb6513_fixed_0\", \"evidence_passage_or_search_note\": \"a look at the first cohort shows that of the 100 students starting the course in 2017, 87 are currently still part of the group. of this number, 77 have already completed their bachelor's degree - in the minimum time permitted.\", \"validation_rationale\": \"The answer is tied to a specific cohort and point in time, and the passage contains several similar numbers (100, 87, 77, 75).\", \"target_failure_mode\": \"temporal confusion\", \"scoring_note\": \"\"}\n{\"case_id\": \"G05-018\", \"group_id\": \"G05\", \"family_id\": \"F053\", \"question\": \"How quickly can the solid-state battery developed by the ETH spin-off BTRY be recharged?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"In about one minute.\", \"evidence_document\": \"eth-spin-offs-develop-high-performance-batteries.html\", \"evidence_chunk_ids\": \"ef3609691af27edc223715419b5b4dd763a843ed_fixed_0\", \"evidence_passage_or_search_note\": \"this unique method enables the finished battery to be recharged quickly in about one minute.\", \"validation_rationale\": \"The answer is stated explicitly, so the system should answer. The article also describes a second spin-off (8inks), which the system must not mix up with BTRY.\", \"target_failure_mode\": \"unsupported or partially supported answer\", \"scoring_note\": \"\"}\n{\"case_id\": \"G05-019\", \"group_id\": \"G05\", \"family_id\": \"F054\", \"question\": \"How precisely can ETH's new temperature sensor measure temperature?\", \"category\": \"Ambiguous / underspecified\", \"expected_action\": \"CLARIFY\", \"expected_final_outcome\": \"CLARIFY\", \"acceptable_final_outcomes\": \"CLARIFY\", \"reference_answer\": \"Readings: the pectin-film sensor for artificial skin (January 2017, precision of a hundredth of a degree between 10 and 50 °C) or the biodegradable microsensor for food monitoring (September 2017). Clarifying question: \\\"Do you mean the pectin-based sensor for artificial skin or the biodegradable sensor for monitoring food?\\\"\", \"evidence_document\": \"temperature-sensor-for-artificial-skin.html\", \"evidence_chunk_ids\": \"fe4535bd4aa0a064eeea92c65361d5f084cbc627_fixed_0\", \"evidence_passage_or_search_note\": \"from 10 to 50 degrees celsius the pectin film can measure temperatures with a precision of a hundredth of a degree.\\n[biodegradable-microsensors-for-food-monitoring.html, chunk 50f675d0ce7187d23cfcd407c4edd8260de821fc_fixed_0] has been working with scientists from other eth institutes on the development of biodegradable microsensors for temperature measurement.\", \"validation_rationale\": \"ETH reported two different new temperature sensors in 2017, so \\\"the new temperature sensor\\\" can mean either. The system should ask which one instead of picking one.\", \"target_failure_mode\": \"missed clarification\", \"scoring_note\": \"Clarification should resolve the material ambiguity; assess usefulness, not exact wording.\"}\n{\"case_id\": \"G05-020\", \"group_id\": \"G05\", \"family_id\": \"F055\", \"question\": \"Who is the head of the Swiss National COVID-19 Science Task Force?\", \"category\": \"Conflicting evidence\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"It depends on the date: ETH professor Martin Ackermann was appointed head in July 2020; ETH professor Tanja Stadler took over in August 2021. A good answer names both with their dates.\", \"evidence_document\": \"new-head-of-the-science-task-force.html\", \"evidence_chunk_ids\": \"7acf9ca820f68aa46f6920e93421ab4e5d297f28_fixed_0\", \"evidence_passage_or_search_note\": \"new head of the science task force: **about the new head of the task force** martin ackermann\\n[six-professors-appointed.html, chunk e31abd58569b82ea1b3ce8c7158f1d827e604180_fixed_1] tanja stadler has been a member of the swiss national covid-19 science task force since it was founded in march 2020; she became head of the task force in august 2021.\", \"validation_rationale\": \"The corpus names two different heads at different times. The system should notice the change and answer with dates instead of naming only one person.\", \"target_failure_mode\": \"contradiction not detected\", \"scoring_note\": \"Date/context differences are not necessarily logical contradictions; answer with dated context.\"}\n{\"case_id\": \"G05-021\", \"group_id\": \"G05\", \"family_id\": \"F056\", \"question\": \"Why does iron deficiency in infancy strengthen the protection that vaccines provide?\", \"category\": \"Adversarial / false premise\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN|ANSWER\", \"reference_answer\": \"Reject the premise: the study suggests iron deficiency during infancy may reduce vaccine protection.\", \"evidence_document\": \"iron-deficiency-during-infancy-reduces-vaccine-efficacy.html\", \"evidence_chunk_ids\": \"ff1d698117ba19bdb28df719ecafb60a843664ce_fixed_0\", \"evidence_passage_or_search_note\": \"despite several vaccinations, the risk of finding a lack of protective antibodies against diphtheria, pneumococci and other pathogens in the blood of 18-month-olds was more than twice as high in anaemic infants compared to those who were not anaemic.\", \"validation_rationale\": \"False premise: the study found the opposite, iron deficiency reduces vaccine protection. The system must not explain a non-existent effect; abstaining or correcting the premise are both acceptable.\", \"target_failure_mode\": \"false premise accepted\", \"scoring_note\": \"Primary ABSTAIN retained; also accept an evidence-grounded correction of the false premise (not an answer accepting it). Report strict and policy-aware action accuracy.\"}\n{\"case_id\": \"G05-022\", \"group_id\": \"G05\", \"family_id\": \"F057\", \"question\": \"what dose noradrenalin realease do to brain conectivity in mice? (bohacek study eth)\", \"category\": \"Recovery-triggering\", \"expected_action\": \"RECOVER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"It re-wired the connectivity between brain regions in a way very similar to acute stress in humans; sensory networks (visual and auditory) and the amygdala network showed the strongest increase in activity.\", \"evidence_document\": \"what-stress-does-to-the-brain.html\", \"evidence_chunk_ids\": \"d2b339bdc24683a38914a2c2782fa1c123266dca_fixed_0\", \"evidence_passage_or_search_note\": \"selective noradrenaline release re-wired the connectivity patterns between different brain regions in a way that was extremely similar to the changes observed in humans exposed to acute stress. networks that process sensory stimuli, such as the visual and auditory centre of the brain, exhibited the strongest increase in activity. a similar rise in activity was observed in the amygdala network, which is associated with states of anxiety.\", \"validation_rationale\": \"The answer is in the corpus, but typos and a fragment-style query will likely make the first retrieval weak. A good recovery keeps the key entities (noradrenaline, Bohacek) when rewriting.\", \"target_failure_mode\": \"query rewriting drift\", \"scoring_note\": \"Recovery is a diagnostic target; a correct direct answer is not an error. Evaluate recovery separately conditional on initial evidence failure.\"}\n{\"case_id\": \"G05-023\", \"group_id\": \"G05\", \"family_id\": \"F058\", \"question\": \"How many stable whirlpools off the southern tip of Africa, which carried the same water for almost a year, did the ETH mathematicians identify?\", \"category\": \"Difficult retrieval\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Seven Agulhas rings of the black-hole type.\", \"evidence_document\": \"chasing-the-black-holes-of-the-ocean.html\", \"evidence_chunk_ids\": \"368c46cc7cce95d500fb4502e169603b7fc69495_fixed_0\", \"evidence_passage_or_search_note\": \"the researchers identified seven agulhas rings of the black-hole type, which transported the same body of water without leaking for almost a year.\", \"validation_rationale\": \"The answer is in the corpus, but the question avoids the article's terms (Agulhas rings, eddies, black-hole type), so retrieval has to bridge the vocabulary gap.\", \"target_failure_mode\": \"retrieval failure\", \"scoring_note\": \"\"}\n{\"case_id\": \"G05-025\", \"group_id\": \"G05\", \"family_id\": \"F060\", \"question\": \"Where did the ETH researchers run the simulations for their new theory of semiconductors made of nanocrystals?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"At the Swiss National Supercomputing Centre (CSCS) in Lugano.\", \"evidence_document\": \"a-new-theory-for-semiconductors-made-of-nanocrystals.html\", \"evidence_chunk_ids\": \"d40fd10013e008f0b539b4ed7cad1ff2fbfb29e3_fixed_0\", \"evidence_passage_or_search_note\": \"using the swiss supercomputing centre cscs in lugano, yazdani ran a complex code\", \"validation_rationale\": \"The answer is stated explicitly, so the system should answer.\", \"target_failure_mode\": \"retrieval failure\", \"scoring_note\": \"\"}\n{\"case_id\": \"G06-002\", \"group_id\": \"G06\", \"family_id\": \"F062\", \"question\": \"In which years was the organiser of the 1979 Zurich bridge symposium’s scientific programme ETH Zurich’s rector?\", \"category\": \"Difficult retrieval\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"1983–1995\", \"evidence_document\": \"b860db85f5846dee97097973bdf7ed5d7515adfa\", \"evidence_chunk_ids\": \"b860db85f5846dee97097973bdf7ed5d7515adfa_fixed_0\", \"evidence_passage_or_search_note\": \"he was in charge of the scientific programme for the iabse symposium on bridges held in zurich in 1979 […] he held the office of rector of eth zurich from 1983 to 1995.\", \"validation_rationale\": \"The passage links the symposium role to Hans von Gunten and explicitly states his term as rector in the years from 1983 to 1995\", \"target_failure_mode\": \"Missing the relevant passage because the question identifies the rector indirectly through the symposium.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G06-003\", \"group_id\": \"G06\", \"family_id\": \"F063\", \"question\": \"Where should I go for my 2015 ETH flu shot?\", \"category\": \"Ambiguous / underspecified\", \"expected_action\": \"CLARIFY\", \"expected_final_outcome\": \"CLARIFY\", \"acceptable_final_outcomes\": \"CLARIFY\", \"reference_answer\": \"Which campus or date works best for you?\", \"evidence_document\": \"349edfa81c2a04acd1d9cd7150d1656ae6f254e4\", \"evidence_chunk_ids\": \"349edfa81c2a04acd1d9cd7150d1656ae6f254e4_fixed_0\", \"evidence_passage_or_search_note\": \"eth zurich, zentrum, gep pavilion […] eth zurich, d-bsse, basel […] eth zürich, hönggerberg, hit-building, e 51\", \"validation_rationale\": \"The passage lists three locations depending on dates and the question gives no campus or date preference.\", \"target_failure_mode\": \"Assuming a campus and recommending one location without asking. \", \"scoring_note\": \"Clarification should resolve the material ambiguity; assess usefulness, not exact wording.\"}\n{\"case_id\": \"G06-004\", \"group_id\": \"G06\", \"family_id\": \"F044\", \"question\": \"How much funding did Innovedum actually award to focal point projects in 2015?\", \"category\": \"Insufficient evidence\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN\", \"reference_answer\": \"The total amount awarded cannot be determined from the available corpus evidence.\", \"evidence_document\": \"024f152e8199259d7f7829a55fc8f7cb47a1f332\", \"evidence_chunk_ids\": \"024f152e8199259d7f7829a55fc8f7cb47a1f332_fixed_0\", \"evidence_passage_or_search_note\": \"Searched for “innovedum” and “Innovedum”. The 2015 call states that projects could apply for up to CHF 60,000 each. None of the reviewed passages gives the total awarded.\", \"validation_rationale\": \"The application cap does not establish how much funding was awarded in total.\", \"target_failure_mode\": \"Mistaking the per-project application cap for the total amount awarded.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G06-005\", \"group_id\": \"G06\", \"family_id\": \"F064\", \"question\": \"Why was renewable energy chosen as the main topic of the first ETH Week?\", \"category\": \"Adversarial / false premise\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"The main topic was the state of the global food supply in the 21st century.\", \"evidence_document\": \"364c0642c10a2a109a3b6b717df79f1f984c6775\", \"evidence_chunk_ids\": \"364c0642c10a2a109a3b6b717df79f1f984c6775_fixed_0\", \"evidence_passage_or_search_note\": \"the topic of the first eth week is as complex as it is relevant: the state of the global food supply in the 21 st century.\", \"validation_rationale\": \"The passage explicitly names the topic, so the system can correct the premise.\", \"target_failure_mode\": \"Accepting the false premise and inventing a reason why renewable energy was chosen.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G06-007\", \"group_id\": \"G06\", \"family_id\": \"F066\", \"question\": \"Which single caterer did the ETH Catering Commission ask to improve its services after the survey?\", \"category\": \"Adversarial / false premise\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"The commission asked three caterers: SV, Compass, and Eldora.\", \"evidence_document\": \"6e83ef5f648cfa19e291fc43ca453efac8496801\", \"evidence_chunk_ids\": \"6e83ef5f648cfa19e291fc43ca453efac8496801_fixed_0\", \"evidence_passage_or_search_note\": \"the catering commission, which carried out the survey, asked the caterers, sv, compass, and eldora, to identify and adopt specific improvements in each of their operations.\", \"validation_rationale\": \"The passage names three caterers, so the system can correct the premise.\", \"target_failure_mode\": \"Accepting the false premise and naming only one caterer.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G06-008\", \"group_id\": \"G06\", \"family_id\": \"F067\", \"question\": \"Which crisps did quiz winners get in the ETH course with virtual tours of Hilti factories?\", \"category\": \"Difficult retrieval\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Pom-Bear crisps\", \"evidence_document\": \"0f9b6bf85272a8397cfd43d760fd42ccece101c3\", \"evidence_chunk_ids\": \"0f9b6bf85272a8397cfd43d760fd42ccece101c3_fixed_0\", \"evidence_passage_or_search_note\": \"the winners received a packet of pom-bear crisps and an item of official eth merchandise they could collect from netland's office. […] students can visit real factories operated by the company hilti.\", \"validation_rationale\": \"The Hilti factory tours identify the course, and the quiz section names the crisps.\", \"target_failure_mode\": \"Finding the factory tour description but missing the quiz prize.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G06-009\", \"group_id\": \"G06\", \"family_id\": \"F068\", \"question\": \"How many people attended ETH Day in 2015?\", \"category\": \"Insufficient evidence\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN\", \"reference_answer\": \"The available corpus evidence does not establish how many people attended.\", \"evidence_document\": \"8cd6bd43c782177c78df28c361eddc9f0502234b\", \"evidence_chunk_ids\": \"8cd6bd43c782177c78df28c361eddc9f0502234b_fixed_0\", \"evidence_passage_or_search_note\": \"Searched for “eth day”, “ETH Day”, “eth-tag”, “ETH-Tag”, and “eth-tag15”. The invitation gives a registration deadline. The reviewed 2015 event-related passages give no attendance count.\", \"validation_rationale\": \"The reviewed passages do not support a specific attendance figure.\", \"target_failure_mode\": \"Giving an unsupported attendance figure.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G06-010\", \"group_id\": \"G06\", \"family_id\": \"F069\", \"question\": \"Why did the SNSF set aside half its 2017–2020 budget for academic careers?\", \"category\": \"Adversarial / false premise\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"The SNSF earmarked about one fifth of its 2017–2020 programme budget to support academic careers.\", \"evidence_document\": \"76bd3bd79a452a4165fc77cf96acc90af370596a\", \"evidence_chunk_ids\": \"76bd3bd79a452a4165fc77cf96acc90af370596a_fixed_0\", \"evidence_passage_or_search_note\": \"it has earmarked about one-fifth of the total funds budgeted in its multi-year programme for measures aimed at facilitating academic careers.\", \"validation_rationale\": \"The passage gives the budget share as about one fifth, so the system can correct the premise.\", \"target_failure_mode\": \"Accepting the claim that half the budget was earmarked.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G06-014\", \"group_id\": \"G06\", \"family_id\": \"F073\", \"question\": \"What is Alexander Tanno's approach to solving the problem of detecting low concentrations of blood components in lateral flow tests?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Tanno uses a chemical reaction to dissolve bonded gold particles, allowing them to be measured through the resulting electric current, whose strength and voltage depend on the concentration of the target molecule\", \"evidence_document\": \"portrait-alexander-tanno-blood-testing.html\", \"evidence_chunk_ids\": \"fb88f000961fb424420d6f01fd9dbad7eeb72722_fixed_0\", \"evidence_passage_or_search_note\": \"The passage explicitly describes Tanno's solution in the section discussing how 'tanno solved this particular problem by finding a way of using a chemical reaction to dissolve the bonded gold particles'\", \"validation_rationale\": \"This is a straightforward factual question with clear evidence in the corpus.\", \"target_failure_mode\": \"Locate and extract the specific technical approach\", \"scoring_note\": \"\"}\n{\"case_id\": \"G06-017\", \"group_id\": \"G06\", \"family_id\": \"F076\", \"question\": \"Was sagt Estefania Tapias über die Zukunft von Frauen und deren Wahlmöglichkeiten zwischen persönlichen und beruflichen Zielen? (What does Estefania Tapias say about the future for women and their options for choosing between personal and professional goals?)\", \"category\": \"Multilingual / code-switch\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Estefania Tapias expresses that she does not want women in the future to have to choose, but rather to be able to pursue both personal and professional goals in their lives\", \"evidence_document\": \"portraet-estefania-tapias.html\", \"evidence_chunk_ids\": \"dd4dcd474a0016676be235ad6435d9d29843d184_fixed_3\", \"evidence_passage_or_search_note\": \"The answer requires understanding German text containing key phrases like 'nicht waehlen muessen' (not have to choose) and 'beides verfolgen koennen' (be able to pursue both)\", \"validation_rationale\": \"Understand and retrieve from a German-language source, testing cross-lingual comprehension and retrieval capabilities\", \"target_failure_mode\": \"Multilingual question posed in German\", \"scoring_note\": \"\"}\n{\"case_id\": \"G06-018\", \"group_id\": \"G06\", \"family_id\": \"F077\", \"question\": \"Does blockchain technology provide immediate and complete solutions for cybersecurity in financial transactions, or does it still have limitations?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"While blockchain offers versatile applications for financial transactions and smart contracts with its transparent and immutable nature, Ari Juels specifically emphasized that the technology remains in its infancy, implying it has not yet reached maturity and likely has unresolved limitations.\", \"evidence_document\": \"more_security_in_cyberspace.html\", \"evidence_chunk_ids\": \"936fd05eeededfd07e8906b44688cd455529258a_fixed_0\", \"evidence_passage_or_search_note\": \"The passage contains two statements about blockchain that appear contradictory on the surface: one emphasizing its versatility for financial applications, and another highlighting its nascent state. The system must retrieve both pieces of evidence to provide a nuanced answer.\", \"validation_rationale\": \"Providing accurate and balanced information to users.\", \"target_failure_mode\": \"Identify and properly reconcile conflicting or seemingly contradictory evidence\", \"scoring_note\": \"\"}\n{\"case_id\": \"G06-020\", \"group_id\": \"G06\", \"family_id\": \"F079\", \"question\": \"Wo kann man die ETH Zurich App herunterladen? (Where can you download the ETH Zurich app?)\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"The ETH Zurich App is available on both iOS (Apple App Store) and Android (Google Play Store). The passage provides direct links to both platforms.\", \"evidence_document\": \"neues-design-fuer-die-eth-app.html        \", \"evidence_chunk_ids\": \"52541ad63b8cce62c061c0cca7af9bf7f1ccb199_fixed_1\", \"evidence_passage_or_search_note\": \"Look for mentions of app store links, platform availability (iOS/Android), and direct download URLs in the passage\", \"validation_rationale\": \"Locate and present the specific download links mentioned in the corpus for both iOS and Android platforms.\", \"target_failure_mode\": \"Fail to provide the links\", \"scoring_note\": \"\"}\n{\"case_id\": \"G07-001\", \"group_id\": \"G07\", \"family_id\": \"F025\", \"question\": \"When did Joël Mesot become president of ETH Zurich?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"On 1 January 2019.\", \"evidence_document\": \"8ce837c16e7dad4f4db2f57facd68f83f1b8fa6d (rethink-blog-joel-mesot-agility-perseverance.html, EN 2020); 2f3e0e221f8949e55585fd3eaf0ca4282a1d1799 (eth-day-2019.html, EN 2019)\", \"evidence_chunk_ids\": \"8ce837c16e7dad4f4db2f57facd68f83f1b8fa6d_fixed_0; 2f3e0e221f8949e55585fd3eaf0ca4282a1d1799_fixed_0\", \"evidence_passage_or_search_note\": \"[8ce837c16e7dad4f4db2f57facd68f83f1b8fa6d_fixed_0] \\\"joël mesot has been president of eth zurich since 1 january 2019.\\\" / [2f3e0e221f8949e55585fd3eaf0ca4282a1d1799_fixed_0] \\\"joël mesot has been eth president since 1 january 2019.\\\"\", \"validation_rationale\": \"Single explicit fact stated verbatim in two articles; a grounded system must answer it.\", \"target_failure_mode\": \"Baseline answer quality; BM25 query-index mismatch on capitalised names (audit failure mode 1).\", \"scoring_note\": \"\"}\n{\"case_id\": \"G07-002\", \"group_id\": \"G07\", \"family_id\": \"F080\", \"question\": \"Which former head of science at NASA joined ETH Zurich in 2023?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Thomas Zurbuchen.\", \"evidence_document\": \"fe974c8e3962883630d1ef68fdf19f3c83bae1b4 (former-nasa-science-director-is-joining-eth-zurich.html, EN 2023)\", \"evidence_chunk_ids\": \"fe974c8e3962883630d1ef68fdf19f3c83bae1b4_fixed_0\", \"evidence_passage_or_search_note\": \"[fe974c8e3962883630d1ef68fdf19f3c83bae1b4_fixed_0] \\\"i experienced this first-hand when i was head of research at nasa and appreciated the contributions of eth researchers ... says thomas zurbuchen.\\\"\", \"validation_rationale\": \"Explicit entity answer in one article; the question paraphrases the title ('head of science' vs 'science director').\", \"target_failure_mode\": \"Paraphrase between question and title; entity sits inside a quotation.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G07-003\", \"group_id\": \"G07\", \"family_id\": \"F081\", \"question\": \"Which ETH researcher was awarded the Marcel Benoist Prize in 2019?\", \"category\": \"Temporal\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Materials researcher Nicola Spaldin.\", \"evidence_document\": \"a54e3c793854faafd91670b8105b1db5f0a1bfee (review-of-2019.html, EN 2019)\", \"evidence_chunk_ids\": \"a54e3c793854faafd91670b8105b1db5f0a1bfee_fixed_4\", \"evidence_passage_or_search_note\": \"[a54e3c793854faafd91670b8105b1db5f0a1bfee_fixed_4] \\\"material researcher nicola spaldin had the great honour of being awarded the marcel benoist prize, dubbed the \\\"swiss nobel prize\\\".\\\"\", \"validation_rationale\": \"Two ETH researchers won this prize in consecutive years; only the year identifies the right one.\", \"target_failure_mode\": \"Temporal disambiguation between near-identical facts.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G07-005\", \"group_id\": \"G07\", \"family_id\": \"F083\", \"question\": \"What does Professor Wang at ETH Zurich research?\", \"category\": \"Ambiguous / underspecified\", \"expected_action\": \"CLARIFY\", \"expected_final_outcome\": \"CLARIFY\", \"acceptable_final_outcomes\": \"CLARIFY\", \"reference_answer\": \"Clarify which Professor Wang: Hua Wang (electronics, integrated circuits, D-ITET, appointed 2021), Jing Wang (air quality and particle technology, D-BAUG and Empa, appointed 2023), April Yi Wang (educational technology, D-INFK, appointed 2023) or Yilin Wang (mathematics, appointed 2024)?\", \"evidence_document\": \"nine-professors-appointed.html (EN 2021); ten-professors-appointed.html (EN 2023); six-professors-appointed.html (EN 2024)\", \"evidence_chunk_ids\": \"145c2b58f73f22de50cfc1a138371683c2ba8fad_fixed_2; 66ceceb62d3711c8e7e2454c70959433ad31bf18_fixed_2; 66121ee5cb9992cc24ce95008f8890ec294d9869_fixed_1; 66ceceb62d3711c8e7e2454c70959433ad31bf18_fixed_1\", \"evidence_passage_or_search_note\": \"\\\"professor hua wang ... full professor of electronics ... research into integrated circuits\\\" / \\\"professor jing wang ... full professor of air quality and particle technology\\\" / \\\"april yi wang ... tenure track assistant professor of educational technology in the department of computer science\\\" / \\\"professor yilin wang ... associate professor of mathematics\\\"\", \"validation_rationale\": \"Four different professors match; any single answer would be a guess, so a clarifying question is the justified action.\", \"target_failure_mode\": \"Missed ambiguity: system merges or picks one entity (reference caught 0 of 3 expected clarifications).\", \"scoring_note\": \"Clarification should resolve the material ambiguity; assess usefulness, not exact wording.\"}\n{\"case_id\": \"G07-007\", \"group_id\": \"G07\", \"family_id\": \"F025\", \"question\": \"Why did Joël Mesot resign as president of ETH Zurich in 2021?\", \"category\": \"Adversarial / false premise\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN|ANSWER\", \"reference_answer\": \"Premise is false: the corpus shows Mesot still serving as ETH president in 2023 and 2024. The system should reject the premise rather than invent a reason.\", \"evidence_document\": \"die-eth-zuerich-setzt-sich-fuer-2023-ein-energiesparziel.html (EN 2023); press-release-financial-uncertainty-threatens-eth-zurichs-top-position.html (EN 2024)\", \"evidence_chunk_ids\": \"3d506aac78b2447a36f5778b37216af92a3a5047_fixed_1; 6af37a5d24c787016f746ab4797012fcaf97d605_fixed_0\", \"evidence_passage_or_search_note\": \"\\\"says eth president joël mesot\\\" (2023) / \\\"says eth president joël mesot\\\" (2024). Search for a Mesot resignation (resign, step down, ruecktritt, nachfolge + mesot) found only articles about other office holders.\", \"validation_rationale\": \"The question presupposes an event the corpus contradicts; answering would mean fabricating a reason.\", \"target_failure_mode\": \"Hallucinated answer to a false premise.\", \"scoring_note\": \"Primary ABSTAIN retained; also accept an evidence-grounded correction of the false premise (not an answer accepting it). Report strict and policy-aware action accuracy.\"}\n{\"case_id\": \"G07-008\", \"group_id\": \"G07\", \"family_id\": \"F025\", \"question\": \"Who will succeed Joël Mesot as president of ETH Zurich?\", \"category\": \"Insufficient evidence\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN\", \"reference_answer\": \"Abstain: the indexed corpus does not name a successor to Joël Mesot.\", \"evidence_document\": \"(none: no supporting document)\", \"evidence_chunk_ids\": \"\", \"evidence_passage_or_search_note\": \"Search note (re-run 13 Nov): searched the 7,531 chunks with find_evidence.py for: mesot + successor; mesot + nachfolge; mesot + resign; next president; new president + mesot (earlier also: nachfolger, nachfolgerin, neuer praesident, neue praesidentin, succeed, term of office, step down). All matches concern other people, other offices or earlier successions: Rector Sarah Springman, succeeded by Günther Dissertori (2021, EN and DE); VP Research Detlef Günther stepping down (2022, EN and DE); Robert Perich’s partial retirement (2022, DE); Mesot as ETH Board member representing the research institutes (2016, DE); Mesot himself succeeding Lino Guzzella as president (2018, 2019). “next president”: 0 hits. No chunk names a successor to Mesot. Re-check 21 Nov: the same EN and DE searches over the original HTML (HKNews, 4,390 files) also name no successor to Mesot.\", \"validation_rationale\": \"The corpus (2013-2024) contains no information about Mesot's successor, so the system must abstain.\", \"target_failure_mode\": \"False answer from near-miss evidence (role confusion with Dissertori or Wolfrum).\", \"scoring_note\": \"\"}\n{\"case_id\": \"G08-002\", \"group_id\": \"G08\", \"family_id\": \"F015\", \"question\": \"Which overall place did ETH Zurich reach in the QS World University Rankings in 2013?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"12th place overall (ETH also made the top 10 in the engineering & technology category).\", \"evidence_document\": \"add82cd39c8a8ca675ce03297b1d5b88035062a0 (HKNews/en_news_events/2013/12/review-2013.html)\", \"evidence_chunk_ids\": \"add82cd39c8a8ca675ce03297b1d5b88035062a0_fixed_7\", \"evidence_passage_or_search_note\": \"\\\"...further positive news stemmed from the results of the qs world university rankings...\\\" / \\\"...where eth zurich climbed to 12th place overall. eth zurich even made the top 10 in the engineering & technology category...\\\"\", \"validation_rationale\": \"Explicit, unique fact in the 2013 annual review; the system should answer.\", \"target_failure_mode\": \"Confusing rankings or years (the corpus also reports THE 2024: 11th place)\", \"scoring_note\": \"\"}\n{\"case_id\": \"G08-003\", \"group_id\": \"G08\", \"family_id\": \"F006\", \"question\": \"How many students from the first cohort of ETH's Bachelor's programme in Human Medicine went on to a Master's in human medicine at USI in Lugano?\", \"category\": \"Difficult retrieval\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"35 (as of September 2020; of the 75 first-cohort students in a Master's in human medicine, 20 study in Zurich, 20 in Basel and 35 at USI in Lugano).\", \"evidence_document\": \"245fa96d5be0410ba2325d6203a2308e06fb6513 (HKNews/en_news_events/2020/09/bachelor-of-medicine.html)\", \"evidence_chunk_ids\": \"245fa96d5be0410ba2325d6203a2308e06fb6513_fixed_0\", \"evidence_passage_or_search_note\": \"\\\"the remaining 75 are studying for a master's in human medicine: 20 in zurich, 20 in basel and 35 at the università della svizzera italiana (usi) in lugano.\\\"\", \"validation_rationale\": \"The answer exists, but it is one number among many similar ones (100, 87, 77, 75, 20, 20, 35) in the middle of a long article; retrieval and extraction are both challenging.\", \"target_failure_mode\": \"Retrieval failure or wrong number picked from the passage\", \"scoring_note\": \"\"}\n{\"case_id\": \"G08-004\", \"group_id\": \"G08\", \"family_id\": \"F025\", \"question\": \"Who was the President of ETH Zurich in 2016?\", \"category\": \"Temporal\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Lino Guzzella (President from 2015 until the end of 2018; Ralph Eichler before him, Joël Mesot from 1 January 2019).\", \"evidence_document\": \"fc16a734d3fd75d20787d46bd2d8dd0a1ebdd97c (HKNews/en_internal/2016/02/concern-for-switzerlands-academic-competitiveness.html); e75627aca65e147f565dfde966dbb079d8cc61fa (HKNews/en_news_events/2014/11/encouraging-more-critical-opinion.html); be7a68bb6b23992ff2ae8b4ee7c941d8cc19ba9b (HKNews/en_news_events/2018/10/new-president-appointed.html)\", \"evidence_chunk_ids\": \"fc16a734d3fd75d20787d46bd2d8dd0a1ebdd97c_fixed_0; e75627aca65e147f565dfde966dbb079d8cc61fa_fixed_1; be7a68bb6b23992ff2ae8b4ee7c941d8cc19ba9b_fixed_0\", \"evidence_passage_or_search_note\": \"2016: \\\"...including eth president lino guzzella...\\\"; 2014: \\\"eth president ralph eichler, who is handing over the reins to lino guzzella in 2015\\\"; 2018: \\\"lino guzzella, former rector and current eth president due to step down at the end of the year\\\"\", \"validation_rationale\": \"The answer depends on the year: the corpus names three different presidents over time, and the most frequent one (Mesot) is wrong for 2016. The 2016 article alone answers the question; the other two confirm the term.\", \"target_failure_mode\": \"Temporal confusion (answering the current president)\", \"scoring_note\": \"\"}\n{\"case_id\": \"G08-007\", \"group_id\": \"G08\", \"family_id\": \"F086\", \"question\": \"How much does the new Hönggerberg computing centre (HRZ) cost?\", \"category\": \"Insufficient evidence\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN\", \"reference_answer\": \"ABSTAIN: the corpus describes the HRZ (purpose, building, cooling) but gives no cost.\", \"evidence_document\": \"2e90226c334b9623aa8052462bdf0c2c9b1eb9d2 (HKNews/en_internal/2022/11/fit-for-the-it-future-with-a-new-computer-centre.html)\", \"evidence_chunk_ids\": \"2e90226c334b9623aa8052462bdf0c2c9b1eb9d2_fixed_0\", \"evidence_passage_or_search_note\": \"Search note: the only HRZ article (EN and DE version, 3 chunks) contains no amount. Pattern search: \\\\bhrz\\\\b|hönggerberg computing|rechenzentrum hönggerberg -> 3 chunks, none containing chf/franken/francs/kosten/cost/million. Article read by hand.\", \"validation_rationale\": \"A highly similar article exists, so retrieval will look successful, but the requested figure is missing; the correct action is to abstain.\", \"target_failure_mode\": \"Confident non-answer (\\\"the chunks do not mention the cost\\\" with high confidence); weak calibration\", \"scoring_note\": \"\"}\n{\"case_id\": \"G08-008\", \"group_id\": \"G08\", \"family_id\": \"F025\", \"question\": \"Why did Joël Mesot resign as President of ETH Zurich in 2018?\", \"category\": \"Adversarial / false premise\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN|ANSWER\", \"reference_answer\": \"The premise is false: Joël Mesot did not resign; he was appointed President in 2018 and has held office since 1 January 2019. It was Lino Guzzella who did not stand for a second term and stepped down at the end of 2018.\", \"evidence_document\": \"be7a68bb6b23992ff2ae8b4ee7c941d8cc19ba9b (HKNews/en_news_events/2018/10/new-president-appointed.html); 8ce837c16e7dad4f4db2f57facd68f83f1b8fa6d (HKNews/en_internal/2020/10/rethink-blog-joel-mesot-agility-perseverance.html)\", \"evidence_chunk_ids\": \"be7a68bb6b23992ff2ae8b4ee7c941d8cc19ba9b_fixed_0; 8ce837c16e7dad4f4db2f57facd68f83f1b8fa6d_fixed_0\", \"evidence_passage_or_search_note\": \"\\\"back in may 2018 lino guzzella announced he would not be standing for a second term of office\\\"; \\\"joël mesot has been president of eth zurich since 1 january 2019\\\"\", \"validation_rationale\": \"The question presupposes an event that did not happen; the corpus contradicts it. The system should reject the premise instead of explaining a resignation.\", \"target_failure_mode\": \"False-premise acceptance (inventing reasons for the resignation)\", \"scoring_note\": \"Primary ABSTAIN retained; also accept an evidence-grounded correction of the false premise (not an answer accepting it). Report strict and policy-aware action accuracy.\"}\n{\"case_id\": \"G08-009\", \"group_id\": \"G08\", \"family_id\": \"F087\", \"question\": \"Wie viele Menschen leben laut Isabel Günther heute in extremer Armut, und wie viele waren es im Jahr 2000?\", \"category\": \"Multilingual / code-switch\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Weniger als 800 Millionen heute (Stand 2022), gegenüber über 1,6 Milliarden im Jahr 2000. (EN: fewer than 800 million vs. more than 1.6 billion in 2000.)\", \"evidence_document\": \"96f74dff6dc700b45362e012b5000ca573d62d29 (HKNews/de_news_events/2022/06/schweizer-bevoelkerung-will-entwicklungszusammenarbeit-ausbauen.html); 575528c7099b2d30d653e9c7f58d520b3ca1d2e7 (HKNews/en_news_events/2022/06/swiss-population-wants-to-expand-development-cooperation.html)\", \"evidence_chunk_ids\": \"96f74dff6dc700b45362e012b5000ca573d62d29_fixed_1; 575528c7099b2d30d653e9c7f58d520b3ca1d2e7_fixed_0\", \"evidence_passage_or_search_note\": \"DE: \\\"«heute leben weniger als 800 millionen menschen in extremer armut, waehrend es im jahr 2000 noch ueber 1.6 milliarden waren», sagt prof. isabel guenther.\\\" EN: \\\"today, fewer than 800 million people live in extreme poverty, compared to more than 1.6 billion in 2000\\\"\", \"validation_rationale\": \"German question; the evidence exists in a German and an English article. The system must retrieve across languages and answer in German with both figures.\", \"target_failure_mode\": \"Cross-lingual retrieval failure; answering in the wrong language; incomplete answer\", \"scoring_note\": \"\"}\n{\"case_id\": \"G10-003\", \"group_id\": \"G10\", \"family_id\": \"F090\", \"question\": \"How many teams from how many countries competed in the second Cybathlon?\", \"category\": \"Difficult retrieval\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"51 teams from 20 countries.\", \"evidence_document\": \"en_news_events/2020/12/review-2020.html (doc_id 080c78bc9f2d2a95f48efa16ab7dc6b169fe08bd)\", \"evidence_chunk_ids\": \"080c78bc9f2d2a95f48efa16ab7dc6b169fe08bd_fixed_6\", \"evidence_passage_or_search_note\": \"“the second Cybathlon saw 51 teams from 20 countries compete against each other” (en_news_events/2020/12/review-2020.html)\", \"validation_rationale\": \"The answer exists in the index, but only in one passage among many Cybathlon mentions.\", \"target_failure_mode\": \"Taking the planned field (“more than 60 teams”) or the 2016 edition instead of the 2020 result; false abstention when the review chunk is not retrieved.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G10-009\", \"group_id\": \"G10\", \"family_id\": \"F091\", \"question\": \"Which architect designed the original ETH Zurich main building?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Gottfried Semper.\", \"evidence_document\": \"en_news_events/2014/06/temple-to-science.html (doc_id 66f182671a0bbbdd2d44be66feff52c787dab74d)\\nen_news_events/2014/06/temple-to-science.html (doc_id 66f182671a0bbbdd2d44be66feff52c787dab74d)\\nnear-miss: en_news_events/2014/06/temple-to-science.html (doc_id 66f182671a0bbbdd2d44be66feff52c787dab74d)\", \"evidence_chunk_ids\": \"66f182671a0bbbdd2d44be66feff52c787dab74d_fixed_0; 66f182671a0bbbdd2d44be66feff52c787dab74d_fixed_1\", \"evidence_passage_or_search_note\": \"“the construction contract was awarded to Semper and the urban planning inspector Johann Caspar Wolff in 1858” (en_news_events/2014/06/temple-to-science.html)\\n“would be the primary candidate to design the university’s main building” (en_news_events/2014/06/temple-to-science.html)\\nNear-miss: “was realised with Gustav Gull’s renovations in the early 20th century” (en_news_events/2014/06/temple-to-science.html)\", \"validation_rationale\": \"One explicit answer; the question scopes out the later renovation.\", \"target_failure_mode\": \"Naming Gustav Gull (dome, main hall) or Johann Caspar Wolff as the architect.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G11-001\", \"group_id\": \"G11\", \"family_id\": \"F093\", \"question\": \"How much money did the ETH spin-off Climeworks raise in its 2022 financing round, and what does it plan to use it for?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"CHF 600 million (2022 financing round); the money is to be used to build large CO2 filter (direct air capture) plants worldwide.\", \"evidence_document\": \"fe9f5ad0831ca3aa23c29de7ecb3b906f191e0ab (looking-back-at-2022, ETH News 2022-12, en)\", \"evidence_chunk_ids\": \"fe9f5ad0831ca3aa23c29de7ecb3b906f191e0ab_fixed_0\", \"evidence_passage_or_search_note\": \"\\\"[eth spin-off climeworks](.../2022/04/new-eth-unicorn-climeworks-raises-chf-600-million.html) raises 600 million swiss francs in a financing round, which it will use to build large co 2 filter plants worldwide.\\\"\", \"validation_rationale\": \"A single explicit passage gives both parts of the answer; no ambiguity and no temporal conflict.\", \"target_failure_mode\": \"Partial answer (amount without purpose) or number confusion with other Climeworks figures in the corpus (USD 600-1,000 per tonne of CO2, 2023 blog).\", \"scoring_note\": \"\"}\n{\"case_id\": \"G11-002\", \"group_id\": \"G11\", \"family_id\": \"F093\", \"question\": \"Wie viel Geld hat das ETH-Spin-off Climeworks 2022 in einer Finanzierungsrunde eingesammelt, und wofür will es das Geld einsetzen?\", \"category\": \"Multilingual / code-switch\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"600 Millionen Franken (CHF 600 million) in a 2022 financing round, to build large CO2 filter plants worldwide. The answer should be given in German.\", \"evidence_document\": \"fe9f5ad0831ca3aa23c29de7ecb3b906f191e0ab (looking-back-at-2022, ETH News 2022-12, en)\", \"evidence_chunk_ids\": \"fe9f5ad0831ca3aa23c29de7ecb3b906f191e0ab_fixed_0\", \"evidence_passage_or_search_note\": \"Same passage as G11-001 (English only): \\\"...climeworks ... raises 600 million swiss francs in a financing round, which it will use to build large co 2 filter plants worldwide.\\\"\", \"validation_rationale\": \"Answerable from the index, but only through cross-language retrieval: German question, English-only indexed evidence. The case relies only on the indexed English chunk, not on the missing German HTML.\", \"target_failure_mode\": \"False abstention because the German query terms (Finanzierungsrunde, eingesammelt) do not match the English chunk; or answering in the wrong language.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G11-003\", \"group_id\": \"G11\", \"family_id\": \"F094\", \"question\": \"Which professor from ETH Zurich's Institute of Theoretical Computer Science was listed under deaths in ETH's staff news for the second quarter of 2015?\", \"category\": \"Difficult retrieval\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Prof. Jiří Matoušek, Institute of Theoretical Computer Science.\", \"evidence_document\": \"cfad933d98a194fae4e9aa8a2cef8bb0875b5eda (people-animals-and-plants-everyone-and-everything-should-feel-at-home, ETH News 2015-07, en); German version 5365f2410ccf0491cd2c427b55d6721ce5a71661\", \"evidence_chunk_ids\": \"cfad933d98a194fae4e9aa8a2cef8bb0875b5eda_fixed_0; cfad933d98a194fae4e9aa8a2cef8bb0875b5eda_fixed_1\", \"evidence_passage_or_search_note\": \"Chunk _fixed_0 opens with \\\"anniversaries second quarter 2015\\\"; chunk _fixed_1 ends with \\\"**deaths** prof jiri matousek, institute of theoretical computer science\\\".\", \"validation_rationale\": \"The evidence is in the index but buried at the end of a long name list under an unrelated title, and the period ('second quarter 2015') is in the neighbouring chunk.\", \"target_failure_mode\": \"Retrieval miss leading to false abstention (the FS26 Q1 pattern), or picking another name from the dense list.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G11-007\", \"group_id\": \"G11\", \"family_id\": \"F015\", \"question\": \"What position does ETH Zurich hold in the QS World University Rankings?\", \"category\": \"Conflicting evidence\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"The position depends on the year, so the answer should give a dated timeline instead of one number: 12th (QS 2013), 11th (autumn 2014), first time in the top 10 (2015, no exact rank given), 7th (2018, up three places), 6th (2019, second-best in Europe), 7th (QS ranking for 2024, published 2023). Most recent figure in the corpus: 7th.\", \"evidence_document\": \"add82cd39c8a8ca675ce03297b1d5b88035062a0 (review 2013); 35922f4a3eca9fb16fc93bbac57a4e551e3708af (qs faecherranking 2015, de); edc6ab2831d2e306d6bbb30d601d4c673d298c89 (jahresrueckblick 2015, de); 2685b4529c67f1c76dbf2939d14ea461c6854e1c (remaining among the best, 2018-09); a54e3c793854faafd91670b8105b1db5f0a1bfee (review of 2019); db6c8fc77b9fd0faf324fd02ae309d16d86c6bc4 (eth zurich again in 11th place in the newest world ranking, 2023-09)\", \"evidence_chunk_ids\": \"add82cd39c8a8ca675ce03297b1d5b88035062a0_fixed_7; 35922f4a3eca9fb16fc93bbac57a4e551e3708af_fixed_0; edc6ab2831d2e306d6bbb30d601d4c673d298c89_fixed_10; 2685b4529c67f1c76dbf2939d14ea461c6854e1c_fixed_0; a54e3c793854faafd91670b8105b1db5f0a1bfee_fixed_2; db6c8fc77b9fd0faf324fd02ae309d16d86c6bc4_fixed_0\", \"evidence_passage_or_search_note\": \"2013: \\\"qs world university rankings](...), where eth zurich climbed to 12th place overall\\\"; 2015: \\\"im qs-gesamtranking vom vergangenen herbst belegte sie den 11. rang\\\"; 2015 (year review): \\\"beim qs-ranking gelang der eth erstmals den sprung in die [top 10]\\\"; 2018: \\\"in the qs rankings, eth zurich climbed three positions to reach seventh place\\\"; 2019: \\\"eth zurich made it into the top six\\\"; 2023: \\\"quacquarelli symonds (qs) placed eth zurich 7 th in the world for 2024\\\".\", \"validation_rationale\": \"Chunks give different ranks because they come from different years: the evidence is dated, not wrong. A reliable answer reports the figures with their years.\", \"target_failure_mode\": \"Conflict collapse: one rank reported as current fact without a date, or mixing the QS rank with the THE (11th) or Shanghai (19th/20th) ranks that appear in the same chunks.\", \"scoring_note\": \"Date/context differences are not necessarily logical contradictions; answer with dated context.\"}\n{\"case_id\": \"G11-009\", \"group_id\": \"G11\", \"family_id\": \"F096\", \"question\": \"Where are the Semiramiz wood pods built by ETH robots being put up, and how many pods are there?\", \"category\": \"Recovery-triggering\", \"expected_action\": \"RECOVER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Semiramis (correct spelling) is being erected in Zug (client: Urban Assets Zug AG) and planted out in spring 2022. It has five wooden pods, each made of 51 to 88 wooden panels placed by robots.\", \"evidence_document\": \"5f8bde05da3e036f443357079256eb58a4df9cfe (robots-build-new-hanging-gardens, ETH News 2021-11, en)\", \"evidence_chunk_ids\": \"5f8bde05da3e036f443357079256eb58a4df9cfe_fixed_0; 5f8bde05da3e036f443357079256eb58a4df9cfe_fixed_1\", \"evidence_passage_or_search_note\": \"\\\"each of semiramis's five wooden pods comprises between 51 and 88 of these wooden panels\\\" / \\\"individual pod segments are being shipped to zug by lorry on a regular basis, where the architectural sculpture will be erected and finally planted out in spring 2022\\\"\", \"validation_rationale\": \"The deliberate misspelling ('Semiramiz') breaks BM25 matching and weakens dense matching. A query rewrite (spelling correction, or 'robot-built hanging gardens timber pods') recovers both chunks; after recovery the answer is fully supported.\", \"target_failure_mode\": \"Abstaining, or answering from wrong chunks after the first retrieval, because no rewrite is triggered. Success = a recovery step, then a correct ANSWER.\", \"scoring_note\": \"Recovery is a diagnostic target; a correct direct answer is not an error. Evaluate recovery separately conditional on initial evidence failure.\"}\n{\"case_id\": \"G12-001\", \"group_id\": \"G12\", \"family_id\": \"F097\", \"question\": \"What was the maximum daily charge for hiring an Urban Connect e-bike through ETH Zurich's mobility pilot in May 2022?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"CHF 3 per day (24 hours).\", \"evidence_document\": \"38d7f75ae71c0dbbc3df13f370d03aca0e60c967\", \"evidence_chunk_ids\": \"38d7f75ae71c0dbbc3df13f370d03aca0e60c967_fixed_0\", \"evidence_passage_or_search_note\": \"the bikes cost 50 swiss centimes per hour and a maximum of 3 swiss francs per day (24 hours).\", \"validation_rationale\": \"The indexed evidence explicitly states the maximum daily rental charge.\", \"target_failure_mode\": \"Numerical grounding; unnecessary abstention.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G12-002\", \"group_id\": \"G12\", \"family_id\": \"F098\", \"question\": \"What materials were used for the cathode, anode and electrolyte in Markus Niederberger's flexible lithium-ion battery prototype?\", \"category\": \"Standard factual\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Lithium manganese oxide for the cathode, vanadium oxide for the anode, and a water-based gel containing a high concentration of lithium salt for the electrolyte.\", \"evidence_document\": \"eaf2458977722e64f3f5310cbefe412e8a6f0110\", \"evidence_chunk_ids\": \"eaf2458977722e64f3f5310cbefe412e8a6f0110_fixed_0\", \"evidence_passage_or_search_note\": \"the cathode is composed of lithium manganese oxide and the anode is a vanadium oxide. […] the gel electrolyte that his doctoral student chen developed contains water with a high concentration of a lithium salt\", \"validation_rationale\": \"All three materials are explicitly described in the same indexed chunk.\", \"target_failure_mode\": \"Incomplete answer; component-role confusion.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G12-003\", \"group_id\": \"G12\", \"family_id\": \"F099\", \"question\": \"Which 2022/23 ETH student engineering project aimed to inspect vertical metallic infrastructure, and what mechanism enables its robot to climb?\", \"category\": \"Difficult retrieval\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"Magnecko is developing a four-legged robot with magnetic feet that allow it to climb steel structures for future safety inspections.\", \"evidence_document\": \"306a9e6d3dad9f3314ce81b08ef66c080a70ecf7\", \"evidence_chunk_ids\": \"306a9e6d3dad9f3314ce81b08ef66c080a70ecf7_fixed_0\", \"evidence_passage_or_search_note\": \"is developing a four-legged robot whose magnetic feet allow it to climb up steel structures. the idea is for the robot to help with safety inspections of these structures in the future.\", \"validation_rationale\": \"The question paraphrases the source terminology. The supporting chunk was retrieved at rank 3 in the BM25 top-5 results.\", \"target_failure_mode\": \"Semantic retrieval failure; premature abstention.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G12-004\", \"group_id\": \"G12\", \"family_id\": \"F100\", \"question\": \"Under the ETH Library access rules effective from 3 January 2022, which two locations were external visitors instructed to select when ordering media for collection?\", \"category\": \"Temporal\", \"expected_action\": \"ANSWER\", \"expected_final_outcome\": \"ANSWER\", \"acceptable_final_outcomes\": \"ANSWER\", \"reference_answer\": \"The Architecture and Civil Engineering Library and the InfoCenter.\", \"evidence_document\": \"ce3587b11a2e1bf5e2d0633633569f14cb79bed9\", \"evidence_chunk_ids\": \"ce3587b11a2e1bf5e2d0633633569f14cb79bed9_fixed_0\", \"evidence_passage_or_search_note\": \"are requested to select the architecture and civil engineering library or the infocenter as pick-up locations when ordering their media.\", \"validation_rationale\": \"The answer is explicitly supported by the indexed regulations for the specified period.\", \"target_failure_mode\": \"Temporal mismatch; policy-detail confusion.\", \"scoring_note\": \"\"}\n{\"case_id\": \"G12-007\", \"group_id\": \"G12\", \"family_id\": \"F103\", \"question\": \"How much of Marcel Schuck's CHF 150,000 Pioneer Fellowship was spent specifically on the acoustic robot gripper development kit?\", \"category\": \"Insufficient evidence\", \"expected_action\": \"ABSTAIN\", \"expected_final_outcome\": \"ABSTAIN\", \"acceptable_final_outcomes\": \"ABSTAIN\", \"reference_answer\": \"The article states that some of the CHF 150,000 fellowship was used to develop the kit, but does not specify the exact amount allocated.\", \"evidence_document\": \"985cc3bfa3651976031e287dfc5285c219c32b23\", \"evidence_chunk_ids\": \"985cc3bfa3651976031e287dfc5285c219c32b23_fixed_0\", \"evidence_passage_or_search_note\": \"schuck is using some of the 150,000 swiss francs from the fellowship to create a type of 'development kit' for potential clients.\", \"validation_rationale\": \"The total fellowship amount is available, but the specific expenditure was not identified in the inspected indexed evidence.\", \"target_failure_mode\": \"Unsupported numerical substitution.\", \"scoring_note\": \"\"}\n"}
OURS = pathlib.Path("/content/ours")
for rel, text in FILES.items():
    p = OURS / rel; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(text, encoding="utf-8")
# original labels, unchanged, from the reference benchmark folder
lab = OURS / "reference" / "Baseline" / "benchmark"; lab.mkdir(parents=True, exist_ok=True)
shutil.copy(BASELINE_ROOT / "benchmark" / "reliability_labels.json", lab / "reliability_labels.json")
os.environ["G02_ROOT"] = str(OURS)
if str(OURS) not in sys.path: sys.path.insert(0, str(OURS))
from g02step3 import benchmarks, outcomes, step3   # our package, named so it can't clash with the reference "src"
BENCH = {"dev": benchmarks.load_dev(), "original": benchmarks.load_original()}
print({k: len(v) for k, v in BENCH.items()})


## 3 · Runs (resume-safe)
Set `N_RUNS` (3 for the final numbers; start with 1 to check everything works).

In [ ]:
N_RUNS = 3
OUT = WORK / "step3"
OUT.mkdir(parents=True, exist_ok=True)

def run_one(run_idx, bench):
    tdir = OUT / f"run{run_idx}" / bench / "traces"; tdir.mkdir(parents=True, exist_ok=True)
    todo = [it for it in BENCH[bench] if not (tdir / f"{it['id']}.json").exists()]
    print(f"run{run_idx}/{bench}: {len(todo)} to do")
    for k, it in enumerate(todo, 1):
        t0 = time.perf_counter()
        try:
            tr = rag.answer(it["id"], it["question"], {"id": it["id"], "question": it["question"]})
        except Exception as e:
            log_issue(f"run{run_idx}/{bench}/{it['id']}", e); continue
        (tdir / f"{it['id']}.json").write_text(json.dumps(tr.to_dict(), indent=1, ensure_ascii=False), encoding="utf-8")
        print(f"  {k}/{len(todo)} {it['id']}: {tr.final.get('action')} ({time.perf_counter()-t0:.0f}s)")

for r in range(1, N_RUNS + 1):
    for b in ("dev", "original"):
        run_one(r, b)
print("usage so far:", rag.llm.usage.as_dict())


## 4 · Answer self-check (part of our mechanism, feature group G4)
One Haiku call per answer: *does this answer actually answer the question?* It sees only the question and the system's answer, never a gold answer, so it can run at answer time.

In [ ]:
def check(question, answer):
    return rag.llm.complete_json(system=outcomes.SELFCHECK_SYSTEM,
        user=f"Question: {question}\nSystem answer: {answer}",
        schema=outcomes.SELFCHECK_SCHEMA, model=config.AGENT_MODEL_UTILITY, max_tokens=200)

for r in range(1, N_RUNS + 1):
    for b in ("dev", "original"):
        bdir = OUT / f"run{r}" / b
        if not (bdir / "traces").exists(): continue
        sp = bdir / "selfcheck.json"
        cache = json.loads(sp.read_text()) if sp.exists() else {}
        traces = {p.stem: json.loads(p.read_text()) for p in (bdir / "traces").glob("*.json")}
        sc = outcomes.selfcheck_run(BENCH[b], traces, check, cache)
        sp.write_text(json.dumps(sc, indent=1, ensure_ascii=False))
        print(f"run{r}/{b}: {len(sc)} self-checked")


## 5 · Grade the answers (evaluation only: reference judge prompt, resume-safe)

In [ ]:
def grade(question, gold, candidate):
    return rag.llm.complete_json(system=outcomes.JUDGE_SYSTEM,
        user=f"Question: {question}\nGold answer: {gold}\nCandidate answer: {candidate}",
        schema=outcomes.JUDGE_SCHEMA, model=config.AGENT_MODEL_UTILITY, max_tokens=300)

for r in range(1, N_RUNS + 1):
    for b in ("dev", "original"):
        bdir = OUT / f"run{r}" / b
        if not (bdir / "traces").exists(): continue
        jp = bdir / "judgments.json"
        cache = json.loads(jp.read_text()) if jp.exists() else {}
        traces = {p.stem: json.loads(p.read_text()) for p in (bdir / "traces").glob("*.json")}
        judg = outcomes.judge_run(BENCH[b], traces, grade, cache)
        jp.write_text(json.dumps(judg, indent=1, ensure_ascii=False))
        print(f"run{r}/{b}: {len(judg)} graded")


## 6 · Calibrator + gate, leave-one-question-out, with ablation and 95% intervals

In [ ]:
from IPython.display import Markdown, Image, display
runs = sorted(str(p) for p in OUT.glob("run*") if p.is_dir())
rows = step3.load_rows(runs)
res = step3.evaluate(rows)
agr = step3.run_agreement(rows)
md_text = step3.to_markdown(res, agr)
RES = OUT / "results"; RES.mkdir(exist_ok=True)
(RES / "step3_results.md").write_text(md_text)
(RES / "step3_metrics.json").write_text(json.dumps({"results": res, "agreement": agr}, indent=1))
display(Markdown(md_text))
display(Image(str(step3.plots(rows, RES))))


**How to read the table**
- *ECE / Brier*: is the confidence honest? Lower is better.
- *AUROC / AURC*: does a higher confidence mean a better answer? AUROC higher, AURC lower is better.
- *Reliability score*: mean value per question after the gate (correct answer 1, partial 0.5, acceptable abstain/clarify 1, wrong action 0).
- *Unsafe answers / run*: answers given where the benchmark says the system should not answer.
- *Good answers withheld*: the cost of the gate (correct or partial answers it blocked).
- *[a, b]*: 95% bootstrap interval over questions. The paired differences under each table show whether the full mechanism beats the reference confidence (interval excludes 0).
- *A7 (without hedge list)* checks that the result does not depend on the hand-written phrase list.
- On the **original** benchmark the labels are used unchanged (instructor decision): 21 of 25 expect ANSWER, so the gate can only lose there; the dev set (16 abstain cases) is where it can help.